# Figure generation

Generation of architecture and qualitative result figures from the study data, model checkpoints, and recorded prediction artifacts.


In [ ]:
!pip -q install scipy tqdm pillow

In [ ]:

FIGURE_SEED = 42
N_CASES_PER_SPLIT = 3

SELECTION_QUANTILES = [0.10, 0.50, 0.90]

SELECTION_MODE = "quantile"

FIXED_SAMPLE_IDS = {
    "mammography": {
        "validation": [],
        "external": [],
    },
    "mri": {
        "validation": [],
        "external": [],
    },
    "ultrasound": {
        "validation": [],
        "external": [],
    },
}

# Set explicit paths only when automatic discovery is ambiguous.
DATA_ROOT_OVERRIDES = {
    "mammography": "",
    "mri": "",
    "ultrasound": "",
}

# Use a separate checkpoint root for each modality.
CHECKPOINT_ROOT_OVERRIDES = {
    "mammography": "",
    "mri": "",
    "ultrasound": "",
}

RESULTS_ROOT_OVERRIDES = {
    "mammography": "",
    "mri": "",
    "ultrasound": "",
}

OUTPUT_ROOT_OVERRIDE = ""

VERIFY_AGAINST_ARCHIVED_METRICS = True
STRICT_ARCHIVE_VERIFICATION = True
ARCHIVE_DICE_TOLERANCE = 5e-4

GENERATE_ARCHITECTURE_FIGURES = True
GENERATE_INTERNAL_VALIDATION_FIGURES = True
GENERATE_EXTERNAL_FIGURES = True
GENERATE_COMBINED_CONTACT_SHEETS = True
GENERATE_TRAINING_CURVES = True

USE_AMP_FOR_FIGURE_INFERENCE = False

IMAGE_SIZE = 256
NUM_WORKERS = 2

FROZEN_THRESHOLDS = {
    "mammography": {
        "unet": {42: 0.60, 123: 0.60, 2025: 0.75},
        "attention_unet": {42: 0.65, 123: 0.60, 2025: 0.65},
        "swin_tiny_unet": {42: 0.65, 123: 0.65, 2025: 0.60},
    },
    "mri": {
        "unet": {42: 0.80, 123: 0.75, 2025: 0.75},
        "attention_unet": {42: 0.80, 123: 0.65, 2025: 0.75},
        "swin_tiny_unet": {42: 0.70, 123: 0.80, 2025: 0.70},
    },
    "ultrasound": {
        "unet": {42: 0.45, 123: 0.75, 2025: 0.70},
        "attention_unet": {42: 0.40, 123: 0.85, 2025: 0.60},
        "swin_tiny_unet": {42: 0.40, 123: 0.45, 2025: 0.40},
    },
}

MODEL_NAMES = [
    "unet",
    "attention_unet",
    "swin_tiny_unet",
]

MODEL_LABELS = {
    "unet": "U-Net",
    "attention_unet": "Attention U-Net",
    "swin_tiny_unet": "Swin-Tiny U-Net",
}

SPLIT_REGISTRY = {
    "mammography": {
        "validation": "validation",
        "external": "external_inbreast",
    },
    "mri": {
        "validation": "validation",
        "external": "external",
    },
    "ultrasound": {
        "validation": "validation",
        "external": "external",
    },
}

## Required inputs

The figure workflow expects the prepared ROI manifests and relevant validation/held-out NPZ crops for mammography, DCE-MRI, and ultrasound, together with the selected model checkpoints. Explicit paths can be set in the configuration cell.


In [ ]:
from __future__ import annotations

import gc
import hashlib
import io
import json
import math
import os
import random
import re
import shutil
import sys
import time
import zipfile
from collections import defaultdict
from contextlib import nullcontext
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageDraw
from scipy.ndimage import binary_erosion, distance_transform_edt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

IN_KAGGLE = (
    Path("/kaggle/input").exists()
    and Path("/kaggle/working").exists()
)

IN_COLAB = False
if not IN_KAGGLE:
    try:
        from google.colab import drive
        IN_COLAB = True
    except Exception:
        IN_COLAB = False

if IN_COLAB:
    try:
        drive.mount("/content/drive")
    except Exception as error:
        print("Google Drive mount warning:", error)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

RUNTIME_NAME = (
    "kaggle"
    if IN_KAGGLE
    else ("colab" if IN_COLAB else "local")
)

if OUTPUT_ROOT_OVERRIDE:
    OUTPUT_ROOT = Path(OUTPUT_ROOT_OVERRIDE)
elif IN_KAGGLE:
    OUTPUT_ROOT = Path(
        "/kaggle/working/AUTHENTIC_ARTICLE_FIGURES"
    )
elif IN_COLAB:
    OUTPUT_ROOT = Path(
        "/content/drive/MyDrive/"
        "AUTHENTIC_ARTICLE_FIGURES"
    )
else:
    OUTPUT_ROOT = Path(
        "./AUTHENTIC_ARTICLE_FIGURES"
    )

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

ARCHITECTURE_DIR = OUTPUT_ROOT / "architectures"
INTERNAL_DIR = OUTPUT_ROOT / "internal_validation"
EXTERNAL_DIR = OUTPUT_ROOT / "external_validation"
CURVES_DIR = OUTPUT_ROOT / "training_curves"
REPORT_DIR = OUTPUT_ROOT / "provenance"

for directory in [
    ARCHITECTURE_DIR,
    INTERNAL_DIR,
    EXTERNAL_DIR,
    CURVES_DIR,
    REPORT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

print({
    "runtime": RUNTIME_NAME,
    "device": str(DEVICE),
    "torch": torch.__version__,
    "output_root": str(OUTPUT_ROOT),
})
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

random.seed(20260731)
np.random.seed(20260731)
torch.manual_seed(20260731)
torch.cuda.manual_seed_all(20260731)

## 1. Artifact discovery


In [ ]:
KNOWN_MANIFESTS = {
    "mammography": "roi_crop_manifest.csv",
    "mri": "roi_mri_manifest.csv",
    "ultrasound": "roi_us_manifest.csv",
}

MINIMUM_FIGURE_NPZ = {
    "mammography": 286,
    "mri": 14360,
    "ultrasound": 528,
}

EXPECTED_FULL_NPZ = {
    "mammography": 1336,
    "mri": 39372,
    "ultrasound": 2127,
}

AUTO_EXTRACT_ROOT = (
    Path("/kaggle/working/AUTHENTIC_FIGURE_AUTO_EXTRACT")
    if IN_KAGGLE
    else OUTPUT_ROOT / "_auto_extract"
)
AUTO_EXTRACT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def sha256_array(
    array: np.ndarray,
) -> str:
    contiguous = np.ascontiguousarray(
        array
    )
    return hashlib.sha256(
        contiguous.tobytes()
    ).hexdigest()

def search_roots() -> List[Path]:
    roots = []

    if IN_KAGGLE:
        roots.extend([
            Path("/kaggle/input"),
            Path("/kaggle/working"),
        ])

    if IN_COLAB:
        roots.extend([
            Path("/content/drive/MyDrive"),
            Path("/content"),
        ])

    roots.extend([
        Path.cwd(),
        Path("/mnt/data"),
    ])

    unique = []
    seen = set()

    for root in roots:
        if not root.exists():
            continue

        try:
            key = str(root.resolve())
        except Exception:
            key = str(root)

        if key not in seen:
            seen.add(key)
            unique.append(root)

    return unique

def safe_extract_zip(
    zip_path: Path,
    destination: Path,
) -> Path:
    marker = destination / ".extraction_complete"

    if marker.exists():
        return destination

    if destination.exists():
        shutil.rmtree(destination)

    destination.mkdir(
        parents=True,
        exist_ok=True,
    )

    with zipfile.ZipFile(zip_path) as archive:
        destination_resolved = destination.resolve()

        for member in archive.infolist():
            target = (
                destination
                / member.filename
            ).resolve()

            if (
                target != destination_resolved
                and destination_resolved
                not in target.parents
            ):
                raise RuntimeError(
                    f"Unsafe ZIP member: {member.filename}"
                )

        archive.extractall(destination)

    marker.write_text(
        str(zip_path),
        encoding="utf-8",
    )

    print(
        "Extracted:",
        zip_path,
        "->",
        destination,
    )
    return destination

def count_npz(
    root: Path,
    limit: Optional[int] = None,
) -> int:
    count = 0

    try:
        for _ in root.rglob("*.npz"):
            count += 1
            if (
                limit is not None
                and count >= limit
            ):
                break
    except Exception:
        pass

    return count

def kaggle_dataset_mount_root(
    path: Path,
) -> Path:
    """
    Return /kaggle/input/<dataset-slug> for any file located inside a Kaggle
    input. This allows NPZ files distributed across several timestamped
    sibling folders to be indexed together.
    """
    try:
        resolved = path.resolve()
    except Exception:
        resolved = path

    parts = resolved.parts

    if (
        len(parts) >= 4
        and parts[1] == "kaggle"
        and parts[2] == "input"
    ):
        return Path(
            "/kaggle/input"
        ) / parts[3]

    return path.parent

def manifest_split_summary(
    manifest_path: Path,
) -> Dict:
    try:
        frame = pd.read_csv(
            manifest_path,
            usecols=lambda column: (
                column in {
                    "split",
                    "dataset",
                    "sample_id",
                    "npz_path",
                }
            ),
            dtype=str,
            keep_default_na=False,
        )
    except Exception as error:
        return {
            "rows": 0,
            "split_counts": {},
            "error": (
                f"{type(error).__name__}: "
                f"{error}"
            )[:500],
        }

    split_counts = {}

    if "split" in frame.columns:
        normalized_split = (
            frame["split"]
            .astype(str)
            .str.strip()
            .str.lower()
        )
        split_counts = (
            normalized_split
            .value_counts()
            .to_dict()
        )

    return {
        "rows": int(len(frame)),
        "split_counts": {
            str(key): int(value)
            for key, value
            in split_counts.items()
        },
        "error": "",
    }

def required_split_rows(
    modality: str,
    split_counts: Dict[str, int],
) -> int:
    validation_split = (
        SPLIT_REGISTRY[
            modality
        ]["validation"]
        .lower()
    )
    external_split = (
        SPLIT_REGISTRY[
            modality
        ]["external"]
        .lower()
    )

    return int(
        split_counts.get(
            validation_split,
            0,
        )
        + split_counts.get(
            external_split,
            0,
        )
    )

def zip_has_dataset(
    zip_path: Path,
    manifest_name: str,
    minimum_npz: int,
) -> bool:
    try:
        with zipfile.ZipFile(zip_path) as archive:
            names = [
                name.replace("\\", "/")
                for name in archive.namelist()
            ]

        has_manifest = any(
            name.endswith(
                manifest_name
            )
            for name in names
        )
        npz_count = sum(
            name.lower().endswith(
                ".npz"
            )
            for name in names
        )

        return (
            has_manifest
            and npz_count >= minimum_npz
        )

    except Exception:
        return False

def candidate_manifest_paths(
    modality: str,
) -> List[Path]:
    manifest_name = KNOWN_MANIFESTS[
        modality
    ]
    override = DATA_ROOT_OVERRIDES[
        modality
    ]

    roots = (
        [Path(override).expanduser()]
        if override
        else search_roots()
    )

    candidates = []
    seen = set()

    for root in roots:
        if not root.exists():
            continue

        direct = root / manifest_name

        if direct.exists():
            paths = [direct]
        else:
            try:
                paths = list(
                    root.rglob(
                        manifest_name
                    )
                )
            except Exception:
                paths = []

        for path in paths:
            try:
                key = str(
                    path.resolve()
                )
            except Exception:
                key = str(path)

            if key not in seen:
                seen.add(key)
                candidates.append(path)

    return candidates

def build_dataset_candidate(
    modality: str,
    manifest_path: Path,
    override_root: Optional[Path],
) -> Dict:
    manifest_root = (
        manifest_path.parent
    )

    if IN_KAGGLE:
        search_root = (
            kaggle_dataset_mount_root(
                manifest_path
            )
        )
    elif (
        override_root is not None
        and override_root.exists()
    ):
        search_root = override_root
    else:
        search_root = manifest_root

    summary = manifest_split_summary(
        manifest_path
    )

    minimum = MINIMUM_FIGURE_NPZ[
        modality
    ]
    full_expected = EXPECTED_FULL_NPZ[
        modality
    ]

    local_npz = count_npz(
        manifest_root,
        limit=full_expected + 1,
    )
    search_npz = count_npz(
        search_root,
        limit=full_expected + 1,
    )

    required_rows = required_split_rows(
        modality,
        summary[
            "split_counts"
        ],
    )

    effective_required = max(
        minimum,
        required_rows,
    )

    supports_required_splits = (
        required_rows >= minimum
    )

    enough_npz = (
        search_npz >= effective_required
    )

    exact_root_bonus = int(
        manifest_root.name.lower()
        in {
            "roi_crops_256_v1",
            "roi_mri_crops_256_v1",
            "roi_us_crops_256_v1",
        }
    )

    score = (
        int(
            supports_required_splits
        ) * 1_000_000
        + min(
            search_npz,
            full_expected,
        ) * 10
        + summary["rows"]
        + local_npz
        + exact_root_bonus * 100
        - len(
            str(manifest_path)
        ) / 1000
    )

    return {
        "modality": modality,
        "manifest_path": manifest_path,
        "manifest_root": manifest_root,
        "search_root": search_root,
        "manifest_rows": summary["rows"],
        "required_split_rows": required_rows,
        "local_npz": local_npz,
        "search_root_npz": search_npz,
        "supports_required_splits": supports_required_splits,
        "enough_npz": enough_npz,
        "score": score,
        "error": summary["error"],
    }

def discover_dataset_layout(
    modality: str,
) -> Dict:
    override_text = DATA_ROOT_OVERRIDES[
        modality
    ]
    override_root = (
        Path(
            override_text
        ).expanduser()
        if override_text
        else None
    )

    manifests = candidate_manifest_paths(
        modality
    )

    if not manifests:
        manifest_name = KNOWN_MANIFESTS[
            modality
        ]
        minimum_npz = MINIMUM_FIGURE_NPZ[
            modality
        ]

        for base in search_roots():
            try:
                archives = list(
                    base.rglob(
                        "*.zip"
                    )
                )
            except Exception:
                archives = []

            for archive_path in archives:
                if zip_has_dataset(
                    archive_path,
                    manifest_name,
                    minimum_npz,
                ):
                    destination = (
                        AUTO_EXTRACT_ROOT
                        / modality
                        / re.sub(
                            r"[^A-Za-z0-9_.-]+",
                            "_",
                            archive_path.stem,
                        )
                    )
                    safe_extract_zip(
                        archive_path,
                        destination,
                    )

        manifests = candidate_manifest_paths(
            modality
        )

    candidates = [
        build_dataset_candidate(
            modality,
            manifest_path,
            override_root,
        )
        for manifest_path in manifests
    ]

    diagnostic = pd.DataFrame([
        {
            "manifest_path": str(
                item[
                    "manifest_path"
                ]
            ),
            "manifest_rows": item[
                "manifest_rows"
            ],
            "required_split_rows": item[
                "required_split_rows"
            ],
            "local_npz": item[
                "local_npz"
            ],
            "search_root": str(
                item[
                    "search_root"
                ]
            ),
            "search_root_npz": item[
                "search_root_npz"
            ],
            "supports_required_splits": item[
                "supports_required_splits"
            ],
            "enough_npz": item[
                "enough_npz"
            ],
            "error": item[
                "error"
            ],
        }
        for item in candidates
    ])

    valid = [
        item
        for item in candidates
        if (
            item[
                "supports_required_splits"
            ]
            and item[
                "enough_npz"
            ]
        )
    ]

    if not valid:
        print(
            f"\nDataset discovery diagnostics for {modality}:"
        )

        if len(diagnostic):
            display(
                diagnostic
            )
        else:
            print(
                "No matching manifest was found."
            )

        raise FileNotFoundError(
            f"A usable {modality} dataset was not found. "
            f"The notebook requires the validation and external NPZ files. "
            f"For sharded Kaggle inputs, all timestamped folders must belong "
            f"to the same dataset input. You may also set "
            f"DATA_ROOT_OVERRIDES['{modality}'] to the top-level Kaggle "
            f"dataset mount, for example "
            f"'/kaggle/input/<dataset-slug>'."
        )

    selected = sorted(
        valid,
        key=lambda item: item[
            "score"
        ],
        reverse=True,
    )[0]

    print(
        f"\nSelected {modality} dataset layout:"
    )
    print(
        "  Manifest:",
        selected[
            "manifest_path"
        ],
    )
    print(
        "  Manifest root:",
        selected[
            "manifest_root"
        ],
    )
    print(
        "  NPZ search root:",
        selected[
            "search_root"
        ],
    )
    print(
        "  Manifest rows:",
        selected[
            "manifest_rows"
        ],
    )
    print(
        "  Required validation/external rows:",
        selected[
            "required_split_rows"
        ],
    )
    print(
        "  NPZ visible under manifest root:",
        selected[
            "local_npz"
        ],
    )
    print(
        "  NPZ visible under full search root:",
        selected[
            "search_root_npz"
        ],
    )

    if (
        selected[
            "local_npz"
        ]
        < selected[
            "search_root_npz"
        ]
    ):
        print(
            "  Sharded dataset detected: NPZ files are distributed "
            "across sibling folders and will be indexed together."
        )

    return selected

DATA_LAYOUTS = {
    modality: discover_dataset_layout(
        modality
    )
    for modality in [
        "mammography",
        "mri",
        "ultrasound",
    ]
}

DATA_ROOTS = {
    modality: layout[
        "manifest_root"
    ]
    for modality, layout
    in DATA_LAYOUTS.items()
}

DATA_SEARCH_ROOTS = {
    modality: layout[
        "search_root"
    ]
    for modality, layout
    in DATA_LAYOUTS.items()
}

MANIFEST_PATHS = {
    modality: layout[
        "manifest_path"
    ]
    for modality, layout
    in DATA_LAYOUTS.items()
}

In [ ]:

MODALITY_PATH_TOKENS = {
    "mammography": [
        "mamm",
        "roi256",
        "tracka",
        "cbis",
    ],
    "mri": [
        "mri",
        "trackb",
        "mama",
        "duke",
    ],
    "ultrasound": [
        "ultrasound",
        "trackc",
        "roi_us",
        "bus",
    ],
}

def archive_contains_pt(
    path: Path,
) -> bool:
    try:
        with zipfile.ZipFile(path) as archive:
            return any(
                name.lower().endswith(".pt")
                for name in archive.namelist()
            )
    except Exception:
        return False

def extract_checkpoint_archives() -> None:
    for base in search_roots():
        try:
            archives = list(
                base.rglob("*.zip")
            )
        except Exception:
            archives = []

        for archive_path in archives:
            if not archive_contains_pt(
                archive_path
            ):
                continue

            destination = (
                AUTO_EXTRACT_ROOT
                / "checkpoints"
                / re.sub(
                    r"[^A-Za-z0-9_.-]+",
                    "_",
                    archive_path.stem,
                )
            )

            try:
                safe_extract_zip(
                    archive_path,
                    destination,
                )
            except Exception as error:
                print(
                    "Checkpoint archive warning:",
                    archive_path,
                    error,
                )

extract_checkpoint_archives()

def score_modality_path(
    path: Path,
    modality: str,
) -> int:
    normalized = str(path).lower()
    score = 0

    for token in MODALITY_PATH_TOKENS[
        modality
    ]:
        if token in normalized:
            score += 10

    for other in MODALITY_PATH_TOKENS:
        if other == modality:
            continue
        for token in MODALITY_PATH_TOKENS[
            other
        ]:
            if token in normalized:
                score -= 5

    return score

def discover_checkpoint_root(
    modality: str,
) -> Path:
    override = CHECKPOINT_ROOT_OVERRIDES[
        modality
    ]

    if override:
        root = Path(override).expanduser()
        if not root.exists():
            raise FileNotFoundError(
                f"Checkpoint root not found: {root}"
            )
        return root

    candidates = []

    for base in search_roots() + [
        AUTO_EXTRACT_ROOT / "checkpoints"
    ]:
        if not base.exists():
            continue

        try:
            pt_files = list(
                base.rglob("*.pt")
            )
        except Exception:
            pt_files = []

        if not pt_files:
            continue

        groups = defaultdict(list)
        for path in pt_files:
            groups[path.parent].append(path)

        for parent, files in groups.items():
            model_seed_hits = 0
            names = [
                path.name.lower()
                for path in files
            ]

            for model_name in MODEL_NAMES:
                for seed in [42, 123, 2025]:
                    model_token = model_name.lower()
                    if any(
                        model_token in name
                        and f"seed{seed}" in name
                        and "best" in name
                        for name in names
                    ):
                        model_seed_hits += 1

            if model_seed_hits:
                candidates.append(
                    (
                        score_modality_path(
                            parent,
                            modality,
                        ),
                        model_seed_hits,
                        parent,
                    )
                )

    if not candidates:
        raise FileNotFoundError(
            f"No checkpoint root was found for {modality}. "
            f"Set CHECKPOINT_ROOT_OVERRIDES['{modality}'] to the folder "
            "containing the nine or seed-42 best checkpoints."
        )

    candidates = sorted(
        candidates,
        key=lambda item: (
            item[0],
            item[1],
        ),
        reverse=True,
    )

    top = candidates[0]

    tied = [
        item
        for item in candidates
        if item[:2] == top[:2]
    ]

    if len(tied) > 1:
        print(
            f"Ambiguous checkpoint roots for {modality}:"
        )
        for item in tied:
            print(item)
        raise RuntimeError(
            f"Set CHECKPOINT_ROOT_OVERRIDES['{modality}'] explicitly."
        )

    print(
        f"{modality} checkpoint root:",
        top[2],
    )
    return top[2]

CHECKPOINT_ROOTS = {
    modality: discover_checkpoint_root(
        modality
    )
    for modality in [
        "mammography",
        "mri",
        "ultrasound",
    ]
}

def discover_results_root(
    modality: str,
) -> Optional[Path]:
    override = RESULTS_ROOT_OVERRIDES[
        modality
    ]

    if override:
        root = Path(override).expanduser()
        if not root.exists():
            raise FileNotFoundError(
                f"Results root not found: {root}"
            )
        return root

    candidates = []

    for base in search_roots():
        if not base.exists():
            continue
        try:
            csv_files = list(
                base.rglob("*.csv")
            )
        except Exception:
            csv_files = []

        groups = defaultdict(list)
        for path in csv_files:
            groups[path.parent].append(path)

        for parent, files in groups.items():
            score = score_modality_path(
                parent,
                modality,
            )
            useful = sum(
                any(
                    token in path.name.lower()
                    for token in [
                        "detailed_metrics",
                        "crop_level_metrics",
                        "validation_crop_metrics",
                        "external_crop_metrics",
                        "history",
                        "training_log",
                    ]
                )
                for path in files
            )
            if useful:
                candidates.append(
                    (
                        score,
                        useful,
                        parent,
                    )
                )

    if not candidates:
        print(
            f"No archived results root auto-detected for {modality}. "
            "Prediction figures can still be generated, but archived "
            "metric verification may be unavailable."
        )
        return None

    selected = sorted(
        candidates,
        key=lambda item: (
            item[0],
            item[1],
        ),
        reverse=True,
    )[0]

    print(
        f"{modality} results root:",
        selected[2],
    )
    return selected[2]

RESULTS_ROOTS = {
    modality: discover_results_root(
        modality
    )
    for modality in [
        "mammography",
        "mri",
        "ultrasound",
    ]
}

## 2. Exact model definitions


In [ ]:
try:
    from torchvision.models import swin_t
    TORCHVISION_OK = True
except Exception as error:
    swin_t = None
    TORCHVISION_OK = False
    raise RuntimeError(
        "torchvision.models.swin_t is required."
    ) from error

class IdentityNorm(nn.Module):
    def __init__(self, channels: int):
        super().__init__()

    def forward(self, x):
        return x

def make_norm(
    channels: int,
    use_batchnorm: bool,
) -> nn.Module:
    return (
        nn.BatchNorm2d(channels)
        if use_batchnorm
        else IdentityNorm(channels)
    )

class ConvBlock(nn.Module):
    def __init__(
        self,
        in_ch: int,
        out_ch: int,
        use_batchnorm: bool = True,
        conv_bias: bool = False,
    ):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(
                in_ch,
                out_ch,
                3,
                padding=1,
                bias=conv_bias,
            ),
            make_norm(
                out_ch,
                use_batchnorm,
            ),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                out_ch,
                out_ch,
                3,
                padding=1,
                bias=conv_bias,
            ),
            make_norm(
                out_ch,
                use_batchnorm,
            ),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class UNet(nn.Module):
    def __init__(
        self,
        in_ch: int = 3,
        out_ch: int = 1,
        base: int = 32,
        use_batchnorm: bool = True,
        conv_bias: bool = False,
    ):
        super().__init__()
        block = lambda a, b: ConvBlock(
            a,
            b,
            use_batchnorm=use_batchnorm,
            conv_bias=conv_bias,
        )
        self.e1 = block(in_ch, base)
        self.e2 = block(base, base * 2)
        self.e3 = block(base * 2, base * 4)
        self.e4 = block(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)
        self.center = block(base * 8, base * 16)
        self.u4 = nn.ConvTranspose2d(
            base * 16,
            base * 8,
            2,
            2,
        )
        self.d4 = block(base * 16, base * 8)
        self.u3 = nn.ConvTranspose2d(
            base * 8,
            base * 4,
            2,
            2,
        )
        self.d3 = block(base * 8, base * 4)
        self.u2 = nn.ConvTranspose2d(
            base * 4,
            base * 2,
            2,
            2,
        )
        self.d2 = block(base * 4, base * 2)
        self.u1 = nn.ConvTranspose2d(
            base * 2,
            base,
            2,
            2,
        )
        self.d1 = block(base * 2, base)
        self.out = nn.Conv2d(
            base,
            out_ch,
            1,
        )

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        center = self.center(self.pool(e4))
        d4 = self.d4(
            torch.cat(
                [self.u4(center), e4],
                dim=1,
            )
        )
        d3 = self.d3(
            torch.cat(
                [self.u3(d4), e3],
                dim=1,
            )
        )
        d2 = self.d2(
            torch.cat(
                [self.u2(d3), e2],
                dim=1,
            )
        )
        d1 = self.d1(
            torch.cat(
                [self.u1(d2), e1],
                dim=1,
            )
        )
        return self.out(d1)

class AttentionGate(nn.Module):
    def __init__(
        self,
        F_g: int,
        F_l: int,
        F_int: int,
        use_batchnorm: bool = True,
    ):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(
                F_g,
                F_int,
                1,
                bias=True,
            ),
            make_norm(
                F_int,
                use_batchnorm,
            ),
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(
                F_l,
                F_int,
                1,
                bias=True,
            ),
            make_norm(
                F_int,
                use_batchnorm,
            ),
        )
        self.psi = nn.Sequential(
            nn.Conv2d(
                F_int,
                1,
                1,
                bias=True,
            ),
            make_norm(
                1,
                use_batchnorm,
            ),
            nn.Sigmoid(),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        attention = self.relu(
            self.W_g(g)
            + self.W_x(x)
        )
        return x * self.psi(attention)

class AttentionUNet(nn.Module):
    def __init__(
        self,
        in_ch: int = 3,
        out_ch: int = 1,
        base: int = 32,
        use_batchnorm: bool = True,
        conv_bias: bool = False,
    ):
        super().__init__()
        block = lambda a, b: ConvBlock(
            a,
            b,
            use_batchnorm=use_batchnorm,
            conv_bias=conv_bias,
        )
        self.e1 = block(in_ch, base)
        self.e2 = block(base, base * 2)
        self.e3 = block(base * 2, base * 4)
        self.e4 = block(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)
        self.center = block(base * 8, base * 16)
        self.u4 = nn.ConvTranspose2d(
            base * 16,
            base * 8,
            2,
            2,
        )
        self.a4 = AttentionGate(
            base * 8,
            base * 8,
            base * 4,
            use_batchnorm,
        )
        self.d4 = block(base * 16, base * 8)
        self.u3 = nn.ConvTranspose2d(
            base * 8,
            base * 4,
            2,
            2,
        )
        self.a3 = AttentionGate(
            base * 4,
            base * 4,
            base * 2,
            use_batchnorm,
        )
        self.d3 = block(base * 8, base * 4)
        self.u2 = nn.ConvTranspose2d(
            base * 4,
            base * 2,
            2,
            2,
        )
        self.a2 = AttentionGate(
            base * 2,
            base * 2,
            base,
            use_batchnorm,
        )
        self.d2 = block(base * 4, base * 2)
        self.u1 = nn.ConvTranspose2d(
            base * 2,
            base,
            2,
            2,
        )
        self.a1 = AttentionGate(
            base,
            base,
            max(base // 2, 1),
            use_batchnorm,
        )
        self.d1 = block(base * 2, base)
        self.out = nn.Conv2d(
            base,
            out_ch,
            1,
        )

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        center = self.center(self.pool(e4))

        u4 = self.u4(center)
        d4 = self.d4(
            torch.cat(
                [u4, self.a4(u4, e4)],
                dim=1,
            )
        )
        u3 = self.u3(d4)
        d3 = self.d3(
            torch.cat(
                [u3, self.a3(u3, e3)],
                dim=1,
            )
        )
        u2 = self.u2(d3)
        d2 = self.d2(
            torch.cat(
                [u2, self.a2(u2, e2)],
                dim=1,
            )
        )
        u1 = self.u1(d2)
        d1 = self.d1(
            torch.cat(
                [u1, self.a1(u1, e1)],
                dim=1,
            )
        )
        return self.out(d1)

class SwinTinyUNet(nn.Module):
    def __init__(
        self,
        in_ch: int = 3,
        out_ch: int = 1,
        center_channels: int = 512,
        use_batchnorm: bool = True,
        conv_bias: bool = False,
    ):
        super().__init__()
        if (
            not TORCHVISION_OK
            or swin_t is None
        ):
            raise RuntimeError(
                "torchvision.models.swin_t is unavailable."
            )

        self.swin = swin_t(weights=None)

        if in_ch != 3:
            original = self.swin.features[0][0]
            self.swin.features[0][0] = nn.Conv2d(
                in_ch,
                original.out_channels,
                kernel_size=original.kernel_size,
                stride=original.stride,
                padding=original.padding,
                bias=(
                    original.bias is not None
                ),
            )

        self.features = self.swin.features
        block = lambda a, b: ConvBlock(
            a,
            b,
            use_batchnorm=use_batchnorm,
            conv_bias=conv_bias,
        )

        self.center = block(
            768,
            center_channels,
        )
        self.up3 = nn.ConvTranspose2d(
            center_channels,
            384,
            2,
            2,
        )
        self.dec3 = block(
            384 + 384,
            256,
        )
        self.up2 = nn.ConvTranspose2d(
            256,
            192,
            2,
            2,
        )
        self.dec2 = block(
            192 + 192,
            128,
        )
        self.up1 = nn.ConvTranspose2d(
            128,
            96,
            2,
            2,
        )
        self.dec1 = block(
            96 + 96,
            64,
        )
        self.up0 = nn.ConvTranspose2d(
            64,
            32,
            2,
            2,
        )
        self.dec0 = block(
            32,
            32,
        )
        self.up_final = nn.ConvTranspose2d(
            32,
            32,
            2,
            2,
        )
        self.out = nn.Conv2d(
            32,
            out_ch,
            1,
        )

    def _to_nchw(self, x):
        if (
            x.ndim == 4
            and x.shape[1]
            not in [96, 192, 384, 768]
        ):
            return x.permute(
                0,
                3,
                1,
                2,
            ).contiguous()
        return x

    def forward(self, x):
        features = []
        y = x

        for index, layer in enumerate(
            self.features
        ):
            y = layer(y)
            if index in [1, 3, 5, 7]:
                features.append(
                    self._to_nchw(y)
                )

        if len(features) != 4:
            raise RuntimeError(
                f"Expected four Swin features, got {len(features)}."
            )

        f1, f2, f3, f4 = features
        center = self.center(f4)
        d3 = self.dec3(
            torch.cat(
                [self.up3(center), f3],
                dim=1,
            )
        )
        d2 = self.dec2(
            torch.cat(
                [self.up2(d3), f2],
                dim=1,
            )
        )
        d1 = self.dec1(
            torch.cat(
                [self.up1(d2), f1],
                dim=1,
            )
        )
        d0 = self.dec0(
            self.up0(d1)
        )
        output = self.out(
            self.up_final(d0)
        )

        if (
            output.shape[-2:]
            != x.shape[-2:]
        ):
            output = F.interpolate(
                output,
                size=x.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )

        return output

ARTICLE_EXPECTED_PARAMS = {
    "unet": 7_763_041,
    "attention_unet": 7_851_773,
    "swin_tiny_unet": 38_350_819,
}

def default_model_config(
    model_name: str,
) -> Dict:
    if model_name in {
        "unet",
        "attention_unet",
    }:
        return {
            "in_ch": 3,
            "out_ch": 1,
            "base": 32,
            "use_batchnorm": True,
            "conv_bias": False,
        }

    if model_name == "swin_tiny_unet":
        return {
            "in_ch": 3,
            "out_ch": 1,
            "center_channels": 512,
            "use_batchnorm": True,
            "conv_bias": False,
        }

    raise ValueError(model_name)

def build_model(
    model_name: str,
    config: Optional[Dict] = None,
) -> nn.Module:
    parameters = dict(
        default_model_config(
            model_name
        )
    )
    if config:
        parameters.update(config)

    if model_name == "unet":
        return UNet(**parameters)
    if model_name == "attention_unet":
        return AttentionUNet(**parameters)
    if model_name == "swin_tiny_unet":
        return SwinTinyUNet(**parameters)

    raise ValueError(model_name)

def model_parameter_count(
    model: nn.Module,
) -> int:
    return int(
        sum(
            parameter.numel()
            for parameter
            in model.parameters()
        )
    )

def model_input_channels(
    model: nn.Module,
) -> int:
    for module in model.modules():
        if isinstance(
            module,
            nn.Conv2d,
        ):
            return int(
                module.in_channels
            )
    raise RuntimeError(
        "No Conv2d layer was found to infer input channels."
    )

print(
    "Default architecture parameter counts:"
)
for model_name in MODEL_NAMES:
    model = build_model(model_name)
    print(
        MODEL_LABELS[model_name],
        f"{model_parameter_count(model):,}",
    )
    del model

In [ ]:

MAMMOGRAPHY_ARCHITECTURE_FAMILY = "mammography_v2_groupnorm_silu"

class MammoConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, groups=8):
        super().__init__()
        groups = min(groups, out_ch)
        while out_ch % groups != 0 and groups > 1:
            groups -= 1
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(groups, out_ch),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(groups, out_ch),
            nn.SiLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class MammoUNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=32):
        super().__init__()
        self.e1 = MammoConvBlock(in_ch, base)
        self.e2 = MammoConvBlock(base, base*2)
        self.e3 = MammoConvBlock(base*2, base*4)
        self.e4 = MammoConvBlock(base*4, base*8)
        self.b = MammoConvBlock(base*8, base*16)
        self.pool = nn.MaxPool2d(2)
        self.u4 = nn.ConvTranspose2d(base*16, base*8, 2, 2)
        self.d4 = MammoConvBlock(base*16, base*8)
        self.u3 = nn.ConvTranspose2d(base*8, base*4, 2, 2)
        self.d3 = MammoConvBlock(base*8, base*4)
        self.u2 = nn.ConvTranspose2d(base*4, base*2, 2, 2)
        self.d2 = MammoConvBlock(base*4, base*2)
        self.u1 = nn.ConvTranspose2d(base*2, base, 2, 2)
        self.d1 = MammoConvBlock(base*2, base)
        self.out = nn.Conv2d(base, out_ch, 1)
    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(self.pool(e1)); e3 = self.e3(self.pool(e2)); e4 = self.e4(self.pool(e3))
        b = self.b(self.pool(e4))
        d4 = self.d4(torch.cat([self.u4(b), e4], dim=1))
        d3 = self.d3(torch.cat([self.u3(d4), e3], dim=1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], dim=1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], dim=1))
        return self.out(d1)

class MammoAttentionGate(nn.Module):
    def __init__(self, g_ch, x_ch, inter_ch):
        super().__init__()
        self.g = nn.Conv2d(g_ch, inter_ch, 1)
        self.x = nn.Conv2d(x_ch, inter_ch, 1)
        self.psi = nn.Sequential(nn.SiLU(inplace=True), nn.Conv2d(inter_ch, 1, 1), nn.Sigmoid())
    def forward(self, g, x):
        if g.shape[-2:] != x.shape[-2:]:
            g = F.interpolate(g, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return x * self.psi(self.g(g) + self.x(x))

class MammoAttentionUNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=32):
        super().__init__()
        self.e1 = MammoConvBlock(in_ch, base); self.e2 = MammoConvBlock(base, base*2)
        self.e3 = MammoConvBlock(base*2, base*4); self.e4 = MammoConvBlock(base*4, base*8)
        self.b = MammoConvBlock(base*8, base*16); self.pool = nn.MaxPool2d(2)
        self.u4 = nn.ConvTranspose2d(base*16, base*8, 2, 2); self.a4 = MammoAttentionGate(base*8, base*8, base*4); self.d4 = MammoConvBlock(base*16, base*8)
        self.u3 = nn.ConvTranspose2d(base*8, base*4, 2, 2); self.a3 = MammoAttentionGate(base*4, base*4, base*2); self.d3 = MammoConvBlock(base*8, base*4)
        self.u2 = nn.ConvTranspose2d(base*4, base*2, 2, 2); self.a2 = MammoAttentionGate(base*2, base*2, base); self.d2 = MammoConvBlock(base*4, base*2)
        self.u1 = nn.ConvTranspose2d(base*2, base, 2, 2); self.a1 = MammoAttentionGate(base, base, max(base//2,1)); self.d1 = MammoConvBlock(base*2, base)
        self.out = nn.Conv2d(base, out_ch, 1)
    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(self.pool(e1)); e3 = self.e3(self.pool(e2)); e4 = self.e4(self.pool(e3)); b = self.b(self.pool(e4))
        u4 = self.u4(b); d4 = self.d4(torch.cat([u4, self.a4(u4,e4)], dim=1))
        u3 = self.u3(d4); d3 = self.d3(torch.cat([u3, self.a3(u3,e3)], dim=1))
        u2 = self.u2(d3); d2 = self.d2(torch.cat([u2, self.a2(u2,e2)], dim=1))
        u1 = self.u1(d2); d1 = self.d1(torch.cat([u1, self.a1(u1,e1)], dim=1))
        return self.out(d1)

class MammoUpConv(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__(); self.up = nn.ConvTranspose2d(in_ch, out_ch, 2, 2); self.conv = MammoConvBlock(out_ch+skip_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x,skip], dim=1))

class MammoSwinTinyUNet(nn.Module):
    def __init__(self, out_ch=1):
        super().__init__()
        self.in_adapter = nn.Conv2d(1,3,1,bias=False)
        with torch.no_grad(): self.in_adapter.weight.fill_(1.0)
        self.register_buffer("mean", torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))
        self.register_buffer("std", torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))
        self.swin = swin_t(weights=None); self.features = self.swin.features
        self.center = MammoConvBlock(768,512)
        self.dec3 = MammoUpConv(512,384,256); self.dec2 = MammoUpConv(256,192,128); self.dec1 = MammoUpConv(128,96,64)
        self.up0a = nn.ConvTranspose2d(64,32,2,2); self.c0a = MammoConvBlock(32,32)
        self.up0b = nn.ConvTranspose2d(32,16,2,2); self.c0b = MammoConvBlock(16,16)
        self.out = nn.Conv2d(16,out_ch,1)
    def _nchw(self,x):
        return x.permute(0,3,1,2).contiguous() if x.ndim==4 and x.shape[1] not in [96,192,384,768] else x
    def forward(self,x):
        y=self.in_adapter(x); y=(y-self.mean)/self.std; feats=[]
        for i,layer in enumerate(self.features):
            y=layer(y)
            if i in [1,3,5,7]: feats.append(self._nchw(y))
        s1,s2,s3,s4=feats; x=self.center(s4); x=self.dec3(x,s3); x=self.dec2(x,s2); x=self.dec1(x,s1)
        x=self.c0a(self.up0a(x)); x=self.c0b(self.up0b(x))
        if x.shape[-2:] != (IMAGE_SIZE,IMAGE_SIZE): x=F.interpolate(x,size=(IMAGE_SIZE,IMAGE_SIZE),mode="bilinear",align_corners=False)
        return self.out(x)

def mammography_exact_config(model_name: str) -> Dict:
    common = {"family": MAMMOGRAPHY_ARCHITECTURE_FAMILY, "in_ch": 1, "out_ch": 1, "normalization": "GroupNorm", "activation": "SiLU"}
    if model_name in {"unet","attention_unet"}: return {**common, "base": 32, "conv_bias": False}
    if model_name == "swin_tiny_unet": return {**common, "center_channels": 512, "input_adapter": "1_to_3", "imagenet_normalization": True}
    raise ValueError(model_name)

_BASE_BUILD_MODEL = build_model

def build_model(model_name: str, config: Optional[Dict] = None) -> nn.Module:
    config = dict(config or {})
    if config.get("family") == MAMMOGRAPHY_ARCHITECTURE_FAMILY:
        if model_name == "unet": return MammoUNet(in_ch=int(config.get("in_ch",1)), out_ch=int(config.get("out_ch",1)), base=int(config.get("base",32)))
        if model_name == "attention_unet": return MammoAttentionUNet(in_ch=int(config.get("in_ch",1)), out_ch=int(config.get("out_ch",1)), base=int(config.get("base",32)))
        if model_name == "swin_tiny_unet": return MammoSwinTinyUNet(out_ch=int(config.get("out_ch",1)))
    return _BASE_BUILD_MODEL(model_name, config)

print("Exact mammography architecture parameter counts:")
for _name in MODEL_NAMES:
    _model = build_model(_name, mammography_exact_config(_name))
    print(MODEL_LABELS[_name], f"{model_parameter_count(_model):,}")
    del _model


In [ ]:

CHECKPOINT_ALIASES = {
    "unet": [
        "unet",
    ],
    "attention_unet": [
        "attention_unet",
        "attentionunet",
        "att_unet",
        "attunet",
    ],
    "swin_tiny_unet": [
        "swin_tiny_unet",
        "swintinyunet",
        "swin_unet",
        "swinunet",
    ],
}

STATE_DICT_CONTAINER_KEYS = [
    "model_state",
    "model_state_dict",
    "state_dict",
    "best_model_state_dict",
    "best_state_dict",
    "model",
    "net",
    "network",
    "weights",
    "ema_state_dict",
    "ema_model",
]

KEY_PREFIXES = [
    "module.",
    "_orig_mod.",
    "model.",
    "net.",
    "network.",
]

CHECKPOINT_DIAGNOSTICS = []

def normalize_token(text: str) -> str:
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(text).lower(),
    ).strip("_")

def checkpoint_filename_match(
    path: Path,
    model_name: str,
    seed: int,
) -> bool:
    stem = normalize_token(path.stem)
    accepted = set()

    for alias in CHECKPOINT_ALIASES[
        model_name
    ]:
        token = normalize_token(alias)
        accepted.update({
            f"{token}_seed{seed}_best",
            f"{token}_seed_{seed}_best",
            f"{token}_{seed}_best",
            f"{token}_best_seed{seed}",
            f"{token}_best_seed_{seed}",
        })

    return stem in accepted

def is_tensor_state_dict(value) -> bool:
    return (
        isinstance(value, dict)
        and len(value) > 0
        and all(
            isinstance(key, str)
            for key in value.keys()
        )
        and all(
            torch.is_tensor(item)
            or isinstance(
                item,
                torch.nn.Parameter,
            )
            for item in value.values()
        )
    )

def module_candidates(
    checkpoint,
) -> List[Tuple[str, nn.Module]]:
    output = []
    seen = set()

    def add(label, value):
        if not isinstance(
            value,
            nn.Module,
        ):
            return
        object_id = id(value)
        if object_id in seen:
            return
        seen.add(object_id)
        output.append(
            (label, value)
        )

    add("checkpoint", checkpoint)

    if isinstance(checkpoint, dict):
        queue = [
            ("checkpoint", checkpoint, 0)
        ]
        while queue:
            label, value, depth = queue.pop(0)
            if depth >= 3:
                continue
            if not isinstance(value, dict):
                continue

            for key, nested in value.items():
                nested_label = (
                    f"{label}.{key}"
                )
                add(
                    nested_label,
                    nested,
                )
                if isinstance(nested, dict):
                    queue.append(
                        (
                            nested_label,
                            nested,
                            depth + 1,
                        )
                    )

    return output

def state_dict_candidates(
    checkpoint,
) -> List[Tuple[str, Dict[str, torch.Tensor]]]:
    candidates = []
    seen_objects = set()

    def add_candidate(label, value):
        if isinstance(value, nn.Module):
            value = value.state_dict()

        if not is_tensor_state_dict(value):
            return

        object_id = id(value)
        if object_id in seen_objects:
            return

        seen_objects.add(object_id)
        candidates.append(
            (
                label,
                dict(value),
            )
        )

    add_candidate(
        "checkpoint",
        checkpoint,
    )

    for label, module in module_candidates(
        checkpoint
    ):
        add_candidate(
            f"{label}.state_dict",
            module,
        )

    if isinstance(checkpoint, dict):
        for key in STATE_DICT_CONTAINER_KEYS:
            if key in checkpoint:
                add_candidate(
                    key,
                    checkpoint[key],
                )

        queue = [
            ("checkpoint", checkpoint, 0)
        ]

        while queue:
            label, value, depth = queue.pop(0)
            if depth >= 3:
                continue
            if not isinstance(value, dict):
                continue

            for key, nested in value.items():
                nested_label = (
                    f"{label}.{key}"
                )
                add_candidate(
                    nested_label,
                    nested,
                )
                if isinstance(nested, dict):
                    queue.append(
                        (
                            nested_label,
                            nested,
                            depth + 1,
                        )
                    )

    return candidates

def extract_state_dict(checkpoint):
    candidates = state_dict_candidates(
        checkpoint
    )
    if not candidates:
        raise RuntimeError(
            "No tensor state dictionary was found."
        )
    return candidates[0][1]

def scalar_metadata(
    checkpoint,
    keys: Sequence[str],
):
    if not isinstance(
        checkpoint,
        dict,
    ):
        return None

    for key in keys:
        value = checkpoint.get(key)
        if isinstance(
            value,
            (
                str,
                int,
                float,
                bool,
                np.integer,
                np.floating,
            ),
        ):
            return value

    return None

def state_dict_variants(state_dict):
    variants = []
    queue = [
        ("original", state_dict)
    ]
    seen = set()

    while queue:
        label, variant = queue.pop(0)
        key_tuple = tuple(
            variant.keys()
        )
        if key_tuple in seen:
            continue

        seen.add(key_tuple)
        variants.append(
            (label, variant)
        )

        for prefix in KEY_PREFIXES:
            if (
                len(variant) > 0
                and all(
                    key.startswith(prefix)
                    for key in variant
                )
            ):
                stripped = {
                    key[len(prefix):]: value
                    for key, value
                    in variant.items()
                }
                queue.append(
                    (
                        f"{label}|strip:{prefix}",
                        stripped,
                    )
                )

    return variants

def ordered_shape_remap(
    model_state,
    source_state,
):
    if len(model_state) != len(source_state):
        return None

    remapped = {}

    for (
        target_key,
        target_tensor,
    ), (
        source_key,
        source_tensor,
    ) in zip(
        model_state.items(),
        source_state.items(),
    ):
        if (
            tuple(target_tensor.shape)
            != tuple(source_tensor.shape)
        ):
            return None

        remapped[
            target_key
        ] = source_tensor

    return remapped

def infer_state_hints(
    checkpoint,
) -> Dict:
    candidates = state_dict_candidates(
        checkpoint
    )

    tensor_shapes = []

    for _, state in candidates:
        tensor_shapes.extend(
            tuple(tensor.shape)
            for tensor in state.values()
            if torch.is_tensor(tensor)
        )

    input_channels = set()
    base_widths = set()
    center_channels = set()

    for shape in tensor_shapes:
        if len(shape) != 4:
            continue

        out_ch, in_ch, kh, kw = shape

        if (kh, kw) == (3, 3):
            if in_ch in {1, 3}:
                input_channels.add(
                    int(in_ch)
                )
                if out_ch in {
                    8,
                    16,
                    24,
                    32,
                    48,
                    64,
                }:
                    base_widths.add(
                        int(out_ch)
                    )

            if (
                in_ch == 768
                and out_ch in {
                    256,
                    384,
                    512,
                    768,
                }
            ):
                center_channels.add(
                    int(out_ch)
                )

        if (
            (kh, kw) == (4, 4)
            and out_ch == 96
            and in_ch in {1, 3}
        ):
            input_channels.add(
                int(in_ch)
            )

    has_batchnorm = any(
        any(
            token in key
            for token in [
                "running_mean",
                "running_var",
                "num_batches_tracked",
            ]
        )
        for _, state in candidates
        for key in state.keys()
    )

    return {
        "input_channels": sorted(
            input_channels
        ),
        "base_widths": sorted(
            base_widths
        ),
        "center_channels": sorted(
            center_channels
        ),
        "has_batchnorm": bool(
            has_batchnorm
        ),
    }

def candidate_model_configs(
    model_name: str,
    checkpoint,
    modality: Optional[str] = None,
) -> List[Dict]:
    hints = infer_state_hints(
        checkpoint
    )

    in_channels = (
        hints["input_channels"]
        or [1, 3]
    )

    batchnorm_values = (
        [True, False]
        if not hints["has_batchnorm"]
        else [True]
    )

    configs = []

    if modality == "mammography":
        configs.append(mammography_exact_config(model_name))

    if model_name in {
        "unet",
        "attention_unet",
    }:
        bases = (
            hints["base_widths"]
            or [16, 32, 64]
        )

        for in_ch in in_channels:
            for base in bases:
                for use_batchnorm in batchnorm_values:
                    for conv_bias in [False, True]:
                        configs.append({
                            "in_ch": int(in_ch),
                            "out_ch": 1,
                            "base": int(base),
                            "use_batchnorm": bool(
                                use_batchnorm
                            ),
                            "conv_bias": bool(
                                conv_bias
                            ),
                        })

    elif model_name == "swin_tiny_unet":
        centers = (
            hints["center_channels"]
            or [512, 256, 768]
        )

        for in_ch in in_channels:
            for center_channels in centers:
                for use_batchnorm in batchnorm_values:
                    for conv_bias in [False, True]:
                        configs.append({
                            "in_ch": int(in_ch),
                            "out_ch": 1,
                            "center_channels": int(
                                center_channels
                            ),
                            "use_batchnorm": bool(
                                use_batchnorm
                            ),
                            "conv_bias": bool(
                                conv_bias
                            ),
                        })

    unique = []
    seen = set()

    default = default_model_config(
        model_name
    )

    for config in [
        default,
        *configs,
    ]:
        signature = json.dumps(
            config,
            sort_keys=True,
        )
        if signature not in seen:
            seen.add(signature)
            unique.append(config)

    return unique

def validate_forward(
    model: nn.Module,
    in_ch: int,
) -> Tuple[bool, str]:
    try:
        model.eval()
        with torch.no_grad():
            output = model(
                torch.zeros(
                    1,
                    in_ch,
                    IMAGE_SIZE,
                    IMAGE_SIZE,
                )
            )

        if tuple(output.shape) != (
            1,
            1,
            IMAGE_SIZE,
            IMAGE_SIZE,
        ):
            return (
                False,
                f"Unexpected output shape: {tuple(output.shape)}",
            )

        return True, ""

    except Exception as error:
        return (
            False,
            (
                f"{type(error).__name__}: "
                f"{error}"
            )[:700],
        )

def try_model_config(
    model_name: str,
    config: Dict,
    checkpoint,
) -> Dict:
    model = build_model(
        model_name,
        config,
    )
    model_state = model.state_dict()
    errors = []

    for source_label, source_state in state_dict_candidates(
        checkpoint
    ):
        for variant_label, variant in state_dict_variants(
            source_state
        ):
            try:
                model.load_state_dict(
                    variant,
                    strict=True,
                )
                forward_ok, forward_error = validate_forward(
                    model,
                    int(config["in_ch"]),
                )
                if forward_ok:
                    return {
                        "loaded": True,
                        "load_mode": "strict_exact",
                        "state_source": (
                            f"{source_label}|{variant_label}"
                        ),
                        "config": config,
                        "parameter_count": model_parameter_count(
                            model
                        ),
                        "source_tensor_count": len(
                            variant
                        ),
                        "target_tensor_count": len(
                            model_state
                        ),
                        "error": "",
                    }
                errors.append(
                    forward_error
                )
            except RuntimeError as error:
                errors.append(
                    str(error)[:400]
                )

            remapped = ordered_shape_remap(
                model_state,
                variant,
            )

            if remapped is not None:
                try:
                    model.load_state_dict(
                        remapped,
                        strict=True,
                    )
                    forward_ok, forward_error = validate_forward(
                        model,
                        int(config["in_ch"]),
                    )
                    if forward_ok:
                        return {
                            "loaded": True,
                            "load_mode": (
                                "strict_ordered_shape_remap"
                            ),
                            "state_source": (
                                f"{source_label}|{variant_label}"
                            ),
                            "config": config,
                            "parameter_count": model_parameter_count(
                                model
                            ),
                            "source_tensor_count": len(
                                variant
                            ),
                            "target_tensor_count": len(
                                model_state
                            ),
                            "error": "",
                        }
                    errors.append(
                        forward_error
                    )
                except RuntimeError as error:
                    errors.append(
                        str(error)[:400]
                    )

    return {
        "loaded": False,
        "load_mode": "",
        "state_source": "",
        "config": config,
        "parameter_count": model_parameter_count(
            model
        ),
        "source_tensor_count": max(
            (
                len(state)
                for _, state
                in state_dict_candidates(
                    checkpoint
                )
            ),
            default=0,
        ),
        "target_tensor_count": len(
            model_state
        ),
        "error": " || ".join(
            errors[:6]
        ),
    }

def serialized_module_options(
    checkpoint,
    model_name: str,
) -> List[Dict]:
    options = []

    for label, module in module_candidates(
        checkpoint
    ):
        in_ch = model_input_channels(
            module
        )
        forward_ok, forward_error = validate_forward(
            module,
            in_ch,
        )

        if not forward_ok:
            continue

        options.append({
            "loaded": True,
            "load_mode": "serialized_module",
            "state_source": label,
            "config": {
                "serialized_module": True,
                "in_ch": int(in_ch),
            },
            "parameter_count": model_parameter_count(
                module
            ),
            "source_tensor_count": len(
                module.state_dict()
            ),
            "target_tensor_count": len(
                module.state_dict()
            ),
            "error": "",
        })

    return options

def load_selected_model(
    checkpoint_path: Path,
    model_name: str,
    load_info: Dict,
) -> nn.Module:
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    if load_info[
        "load_mode"
    ] == "serialized_module":
        label = load_info[
            "state_source"
        ]

        for candidate_label, module in module_candidates(
            checkpoint
        ):
            if candidate_label == label:
                return module

        raise RuntimeError(
            f"Serialized module {label} was not found during reload."
        )

    config = dict(
        load_info[
            "config"
        ]
    )
    model = build_model(
        model_name,
        config,
    )

    verification = try_model_config(
        model_name,
        config,
        checkpoint,
    )

    if not verification[
        "loaded"
    ]:
        raise RuntimeError(
            "Checkpoint compatibility changed during reload: "
            f"{verification['error']}"
        )

    model = build_model(
        model_name,
        config,
    )
    selected_source = load_info[
        "state_source"
    ]

    for source_label, source_state in state_dict_candidates(
        checkpoint
    ):
        for variant_label, variant in state_dict_variants(
            source_state
        ):
            label = (
                f"{source_label}|{variant_label}"
            )
            if label != selected_source:
                continue

            if load_info[
                "load_mode"
            ] == "strict_exact":
                model.load_state_dict(
                    variant,
                    strict=True,
                )
                return model

            if load_info[
                "load_mode"
            ] == "strict_ordered_shape_remap":
                remapped = ordered_shape_remap(
                    model.state_dict(),
                    variant,
                )
                if remapped is None:
                    raise RuntimeError(
                        "Ordered shape remap is no longer valid."
                    )
                model.load_state_dict(
                    remapped,
                    strict=True,
                )
                return model

    raise RuntimeError(
        f"Selected state source was not found: {selected_source}"
    )

def inspect_checkpoint_structure(
    checkpoint,
) -> Dict:
    top_level_keys = (
        [
            str(key)
            for key in checkpoint.keys()
        ]
        if isinstance(
            checkpoint,
            dict,
        )
        else []
    )

    state_candidates = state_dict_candidates(
        checkpoint
    )
    hints = infer_state_hints(
        checkpoint
    )

    return {
        "checkpoint_type": type(
            checkpoint
        ).__name__,
        "top_level_keys": " | ".join(
            top_level_keys[:30]
        ),
        "state_candidate_count": len(
            state_candidates
        ),
        "state_candidate_labels": " | ".join(
            label
            for label, _
            in state_candidates[:10]
        ),
        "first_state_keys": (
            " | ".join(
                list(
                    state_candidates[0][1].keys()
                )[:12]
            )
            if state_candidates
            else ""
        ),
        "inferred_input_channels": str(
            hints[
                "input_channels"
            ]
        ),
        "inferred_base_widths": str(
            hints[
                "base_widths"
            ]
        ),
        "inferred_center_channels": str(
            hints[
                "center_channels"
            ]
        ),
        "batchnorm_buffers_detected": hints[
            "has_batchnorm"
        ],
    }

def select_checkpoint(
    modality: str,
    model_name: str,
    seed: int,
) -> Tuple[Path, Dict]:
    root = CHECKPOINT_ROOTS[
        modality
    ]

    candidates = [
        path
        for path in root.rglob("*.pt")
        if checkpoint_filename_match(
            path,
            model_name,
            seed,
        )
    ]

    if not candidates:
        candidates = list(
            root.rglob("*.pt")
        )

    compatible = []

    for path in candidates:
        diagnostic_base = {
            "modality": modality,
            "expected_model_name": model_name,
            "expected_seed": seed,
            "checkpoint_path": str(path),
            "filename_exact_match": checkpoint_filename_match(
                path,
                model_name,
                seed,
            ),
        }

        try:
            checkpoint = torch.load(
                path,
                map_location="cpu",
                weights_only=False,
            )
        except Exception as error:
            row = dict(
                diagnostic_base
            )
            row.update({
                "checkpoint_type": "",
                "top_level_keys": "",
                "state_candidate_count": 0,
                "state_candidate_labels": "",
                "first_state_keys": "",
                "inferred_input_channels": "",
                "inferred_base_widths": "",
                "inferred_center_channels": "",
                "batchnorm_buffers_detected": "",
                "config": "",
                "loaded": False,
                "load_mode": "",
                "state_source": "",
                "parameter_count": "",
                "article_parameter_count_match": False,
                "source_tensor_count": 0,
                "target_tensor_count": 0,
                "error": (
                    f"torch.load failed: "
                    f"{type(error).__name__}: {error}"
                )[:1000],
            })
            CHECKPOINT_DIAGNOSTICS.append(
                row
            )
            continue

        structure = inspect_checkpoint_structure(
            checkpoint
        )

        checkpoint_model = scalar_metadata(
            checkpoint,
            [
                "model_name",
                "architecture",
                "architecture_name",
                "arch",
                "network_name",
            ],
        )
        checkpoint_seed = scalar_metadata(
            checkpoint,
            [
                "seed",
                "random_seed",
            ],
        )

        if checkpoint_model is not None:
            accepted_names = {
                normalize_token(alias)
                for alias in CHECKPOINT_ALIASES[
                    model_name
                ]
            }
            if normalize_token(
                checkpoint_model
            ) not in accepted_names:
                row = dict(
                    diagnostic_base
                )
                row.update(
                    structure
                )
                row.update({
                    "config": "",
                    "loaded": False,
                    "load_mode": "",
                    "state_source": "",
                    "parameter_count": "",
                    "article_parameter_count_match": False,
                    "source_tensor_count": 0,
                    "target_tensor_count": 0,
                    "error": (
                        "Scalar model metadata does not match: "
                        f"{checkpoint_model}"
                    ),
                })
                CHECKPOINT_DIAGNOSTICS.append(
                    row
                )
                continue

        if checkpoint_seed is not None:
            try:
                if int(checkpoint_seed) != int(seed):
                    continue
            except Exception:
                continue

        options = []

        for config in candidate_model_configs(
            model_name,
            checkpoint,
            modality=modality,
        ):
            result = try_model_config(
                model_name,
                config,
                checkpoint,
            )

            row = dict(
                diagnostic_base
            )
            row.update(
                structure
            )
            row.update({
                "config": json.dumps(
                    config,
                    sort_keys=True,
                ),
                "loaded": result[
                    "loaded"
                ],
                "load_mode": result[
                    "load_mode"
                ],
                "state_source": result[
                    "state_source"
                ],
                "parameter_count": result[
                    "parameter_count"
                ],
                "article_parameter_count_match": (
                    result[
                        "parameter_count"
                    ]
                    == ARTICLE_EXPECTED_PARAMS[
                        model_name
                    ]
                ),
                "source_tensor_count": result[
                    "source_tensor_count"
                ],
                "target_tensor_count": result[
                    "target_tensor_count"
                ],
                "error": result[
                    "error"
                ],
            })
            CHECKPOINT_DIAGNOSTICS.append(
                row
            )

            if result[
                "loaded"
            ]:
                options.append(
                    result
                )

        for option in serialized_module_options(
            checkpoint,
            model_name,
        ):
            row = dict(
                diagnostic_base
            )
            row.update(
                structure
            )
            row.update({
                "config": json.dumps(
                    option[
                        "config"
                    ],
                    sort_keys=True,
                ),
                "loaded": True,
                "load_mode": option[
                    "load_mode"
                ],
                "state_source": option[
                    "state_source"
                ],
                "parameter_count": option[
                    "parameter_count"
                ],
                "article_parameter_count_match": (
                    option[
                        "parameter_count"
                    ]
                    == ARTICLE_EXPECTED_PARAMS[
                        model_name
                    ]
                ),
                "source_tensor_count": option[
                    "source_tensor_count"
                ],
                "target_tensor_count": option[
                    "target_tensor_count"
                ],
                "error": "",
            })
            CHECKPOINT_DIAGNOSTICS.append(
                row
            )
            options.append(
                option
            )

        if options:
            mode_rank = {
                "strict_exact": 0,
                "strict_ordered_shape_remap": 1,
                "serialized_module": 2,
            }

            options = sorted(
                options,
                key=lambda item: (
                    mode_rank[
                        item[
                            "load_mode"
                        ]
                    ],
                    int(
                        item[
                            "parameter_count"
                        ]
                        != ARTICLE_EXPECTED_PARAMS[
                            model_name
                        ]
                    ),
                    json.dumps(
                        item[
                            "config"
                        ],
                        sort_keys=True,
                    ),
                ),
            )

            best = options[0]
            best_rank = mode_rank[
                best[
                    "load_mode"
                ]
            ]

            tied = [
                option
                for option in options
                if (
                    mode_rank[
                        option[
                            "load_mode"
                        ]
                    ] == best_rank
                    and option[
                        "parameter_count"
                    ] == best[
                        "parameter_count"
                    ]
                    and option[
                        "config"
                    ] != best[
                        "config"
                    ]
                )
            ]

            if tied:
                raise RuntimeError(
                    "Several model configurations are equally compatible "
                    f"with {path}. Review the diagnostic CSV before using "
                    "the checkpoint."
                )

            compatible.append(
                (
                    path,
                    best,
                )
            )

    diagnostic_frame = pd.DataFrame(
        CHECKPOINT_DIAGNOSTICS
    )
    diagnostic_path = (
        REPORT_DIR
        / "checkpoint_compatibility_diagnostics.csv"
    )
    diagnostic_frame.to_csv(
        diagnostic_path,
        index=False,
    )

    if not compatible:
        relevant = diagnostic_frame[
            (
                diagnostic_frame[
                    "modality"
                ] == modality
            )
            & (
                diagnostic_frame[
                    "expected_model_name"
                ] == model_name
            )
            & (
                pd.to_numeric(
                    diagnostic_frame[
                        "expected_seed"
                    ],
                    errors="coerce",
                )
                == int(seed)
            )
        ]

        print(
            f"\nCheckpoint diagnostics for "
            f"{modality}, {model_name}, seed {seed}:"
        )

        display_columns = [
            column
            for column in [
                "checkpoint_path",
                "checkpoint_type",
                "top_level_keys",
                "first_state_keys",
                "inferred_input_channels",
                "inferred_base_widths",
                "config",
                "load_mode",
                "parameter_count",
                "source_tensor_count",
                "target_tensor_count",
                "error",
            ]
            if column in relevant.columns
        ]

        display(
            relevant[
                display_columns
            ]
        )

        raise FileNotFoundError(
            "No checkpoint could be matched to any verified architecture "
            f"configuration for {modality}, {model_name}, seed {seed}. "
            f"See {diagnostic_path}."
        )

    hashes = defaultdict(list)

    for path, load_info in compatible:
        hashes[
            sha256_file(path)
        ].append(
            (
                path,
                load_info,
            )
        )

    if len(hashes) > 1:
        raise RuntimeError(
            "Multiple non-identical compatible checkpoints were found for "
            f"{modality}, {model_name}, seed {seed}: "
            f"{[str(path) for path, _ in compatible]}"
        )

    copies = next(
        iter(
            hashes.values()
        )
    )

    return sorted(
        copies,
        key=lambda item: (
            len(str(item[0])),
            str(item[0]),
        ),
    )[0]

CHECKPOINT_MAP = {}
CHECKPOINT_LOAD_INFO = {}
MODEL_CONFIG_MAP = {}
ACTUAL_PARAMETER_COUNTS = {}

for modality in [
    "mammography",
    "mri",
    "ultrasound",
]:
    for model_name in MODEL_NAMES:
        (
            checkpoint_path,
            load_info,
        ) = select_checkpoint(
            modality,
            model_name,
            FIGURE_SEED,
        )

        key = (
            modality,
            model_name,
            FIGURE_SEED,
        )

        CHECKPOINT_MAP[
            key
        ] = checkpoint_path
        CHECKPOINT_LOAD_INFO[
            key
        ] = load_info
        MODEL_CONFIG_MAP[
            key
        ] = dict(
            load_info[
                "config"
            ]
        )
        ACTUAL_PARAMETER_COUNTS[
            key
        ] = int(
            load_info[
                "parameter_count"
            ]
        )

checkpoint_table = pd.DataFrame([
    {
        "modality": modality,
        "model_name": model_name,
        "seed": seed,
        "checkpoint_path": str(path),
        "checkpoint_sha256": sha256_file(path),
        "size_mb": (
            path.stat().st_size
            / (1024 ** 2)
        ),
        "load_mode": CHECKPOINT_LOAD_INFO[
            (
                modality,
                model_name,
                seed,
            )
        ][
            "load_mode"
        ],
        "state_source": CHECKPOINT_LOAD_INFO[
            (
                modality,
                model_name,
                seed,
            )
        ][
            "state_source"
        ],
        "model_config": json.dumps(
            MODEL_CONFIG_MAP[
                (
                    modality,
                    model_name,
                    seed,
                )
            ],
            sort_keys=True,
        ),
        "actual_parameter_count": ACTUAL_PARAMETER_COUNTS[
            (
                modality,
                model_name,
                seed,
            )
        ],
        "article_expected_parameter_count": ARTICLE_EXPECTED_PARAMS[
            model_name
        ],
        "article_parameter_count_match": (
            ACTUAL_PARAMETER_COUNTS[
                (
                    modality,
                    model_name,
                    seed,
                )
            ]
            == ARTICLE_EXPECTED_PARAMS[
                model_name
            ]
        ),
        "source_tensor_count": CHECKPOINT_LOAD_INFO[
            (
                modality,
                model_name,
                seed,
            )
        ][
            "source_tensor_count"
        ],
        "target_tensor_count": CHECKPOINT_LOAD_INFO[
            (
                modality,
                model_name,
                seed,
            )
        ][
            "target_tensor_count"
        ],
    }
    for (
        modality,
        model_name,
        seed,
    ), path in CHECKPOINT_MAP.items()
]).sort_values([
    "modality",
    "model_name",
])

checkpoint_table.to_csv(
    REPORT_DIR
    / "checkpoints_used.csv",
    index=False,
)

print(
    "Checkpoint and architecture-configuration recovery: PASS"
)
display(
    checkpoint_table
)

mismatched_counts = checkpoint_table[
    ~checkpoint_table[
        "article_parameter_count_match"
    ]
]

if len(mismatched_counts):
    print(
        "\nIMPORTANT: at least one authentic checkpoint uses a parameter "
        "count different from the current manuscript. Update the article "
        "before submission using the values in checkpoints_used.csv."
    )
    display(
        mismatched_counts[
            [
                "modality",
                "model_name",
                "model_config",
                "actual_parameter_count",
                "article_expected_parameter_count",
            ]
        ]
    )

## 3. Original architecture figures generated from the exact code


In [ ]:

def tensor_shape(value):
    if torch.is_tensor(value):
        return tuple(
            int(item)
            for item in value.shape
        )

    if isinstance(
        value,
        (list, tuple),
    ):
        for item in value:
            shape = tensor_shape(item)
            if shape is not None:
                return shape

    return None

def trace_selected_modules(
    model: nn.Module,
    selected_names: Sequence[str],
    input_channels: int,
) -> Dict[str, Tuple[int, ...]]:
    shapes = {}
    handles = []
    modules = dict(
        model.named_modules()
    )

    for name in selected_names:
        if name not in modules:
            continue

        def make_hook(module_name):
            def hook(module, inputs, output):
                shape = tensor_shape(output)
                if shape is not None:
                    shapes[
                        module_name
                    ] = shape
            return hook

        handles.append(
            modules[name].register_forward_hook(
                make_hook(name)
            )
        )

    model.eval()
    with torch.no_grad():
        output = model(
            torch.zeros(
                1,
                input_channels,
                IMAGE_SIZE,
                IMAGE_SIZE,
            )
        )
        shapes["output"] = tuple(
            int(item)
            for item in output.shape
        )

    for handle in handles:
        handle.remove()

    return shapes

def shape_label(shape):
    if len(shape) == 4:
        return (
            f"{shape[1]} ch\n"
            f"{shape[2]}×{shape[3]}"
        )
    return str(shape)

def add_box(
    axis,
    x,
    y,
    width,
    height,
    title,
    subtitle,
):
    patch = FancyBboxPatch(
        (x, y),
        width,
        height,
        boxstyle="round,pad=0.02",
        linewidth=1.2,
        fill=False,
    )
    axis.add_patch(patch)
    axis.text(
        x + width / 2,
        y + height * 0.62,
        title,
        ha="center",
        va="center",
        fontsize=9,
        fontweight="bold",
    )
    axis.text(
        x + width / 2,
        y + height * 0.28,
        subtitle,
        ha="center",
        va="center",
        fontsize=8,
    )

def add_arrow(
    axis,
    start,
    end,
    connectionstyle="arc3",
    linestyle="-",
):
    axis.add_patch(
        FancyArrowPatch(
            start,
            end,
            arrowstyle="->",
            mutation_scale=10,
            linewidth=1.1,
            linestyle=linestyle,
            connectionstyle=connectionstyle,
        )
    )

def save_figure_formats(
    figure,
    base_path: Path,
):
    for suffix in [
        ".png",
        ".pdf",
        ".svg",
    ]:
        figure.savefig(
            base_path.with_suffix(
                suffix
            ),
            dpi=300,
            bbox_inches="tight",
        )

def configuration_groups(
    model_name: str,
) -> List[Tuple[str, List[str], Dict]]:
    groups = defaultdict(list)
    configs = {}

    for modality in [
        "mammography",
        "mri",
        "ultrasound",
    ]:
        key = (
            modality,
            model_name,
            FIGURE_SEED,
        )
        config = MODEL_CONFIG_MAP[
            key
        ]
        signature = json.dumps(
            config,
            sort_keys=True,
        )
        groups[
            signature
        ].append(modality)
        configs[
            signature
        ] = config

    output = []

    for signature, modalities in groups.items():
        label = (
            "shared"
            if len(modalities) == 3
            else "_".join(modalities)
        )
        output.append(
            (
                label,
                modalities,
                configs[
                    signature
                ],
            )
        )

    return output

In [ ]:

def draw_unet_architecture(
    model_name: str,
    config_label: str,
    modalities: Sequence[str],
    config: Dict,
):
    if config.get(
        "serialized_module",
        False,
    ):
        reference_modality = modalities[0]
        model = load_selected_model(
            CHECKPOINT_MAP[
                (
                    reference_modality,
                    model_name,
                    FIGURE_SEED,
                )
            ],
            model_name,
            CHECKPOINT_LOAD_INFO[
                (
                    reference_modality,
                    model_name,
                    FIGURE_SEED,
                )
            ],
        )
    else:
        model = build_model(
            model_name,
            config,
        )

    attention = (
        model_name
        == "attention_unet"
    )

    exact_mammography_family = (config.get("family") == MAMMOGRAPHY_ARCHITECTURE_FAMILY)
    center_module = "b" if exact_mammography_family else "center"

    selected = [
        "e1",
        "e2",
        "e3",
        "e4",
        center_module,
        "d4",
        "d3",
        "d2",
        "d1",
        "out",
    ]

    if attention:
        selected.extend([
            "a4",
            "a3",
            "a2",
            "a1",
        ])

    in_ch = int(
        config.get(
            "in_ch",
            model_input_channels(model),
        )
    )

    shapes = trace_selected_modules(
        model,
        selected,
        input_channels=in_ch,
    )

    if center_module != "center" and center_module in shapes:
        shapes["center"] = shapes[center_module]

    required = {
        "e1",
        "e2",
        "e3",
        "e4",
        "center",
        "d4",
        "d3",
        "d2",
        "d1",
    }

    if not required.issubset(
        shapes.keys()
    ):
        raise RuntimeError(
            "The serialized module does not expose the expected U-Net "
            f"block names. Available traced blocks: {sorted(shapes)}"
        )

    figure, axis = plt.subplots(
        figsize=(15.8, 7.2)
    )
    axis.set_xlim(
        0,
        16.0,
    )
    axis.set_ylim(
        0,
        8,
    )
    axis.axis("off")

    encoder_positions = {
        "e1": (1.2, 6.3),
        "e2": (2.8, 5.1),
        "e3": (4.4, 3.9),
        "e4": (6.0, 2.7),
        "center": (7.6, 1.5),
    }
    decoder_positions = {
        "d4": (9.2, 2.7),
        "d3": (10.8, 3.9),
        "d2": (12.4, 5.1),
        "d1": (14.0, 6.3),
    }

    box_w = 1.2
    box_h = 0.85

    add_box(
        axis,
        0.0,
        6.3,
        box_w,
        box_h,
        "Input",
        f"{in_ch} ch\n256×256",
    )

    previous = (
        box_w,
        6.72,
    )

    for key in [
        "e1",
        "e2",
        "e3",
        "e4",
        "center",
    ]:
        x, y = encoder_positions[
            key
        ]
        add_box(
            axis,
            x,
            y,
            box_w,
            box_h,
            key.upper(),
            shape_label(
                shapes[key]
            ),
        )
        add_arrow(
            axis,
            previous,
            (
                x,
                y + box_h / 2,
            ),
        )
        previous = (
            x + box_w,
            y + box_h / 2,
        )

    previous = (
        encoder_positions[
            "center"
        ][0] + box_w,
        encoder_positions[
            "center"
        ][1] + box_h / 2,
    )

    for key in [
        "d4",
        "d3",
        "d2",
        "d1",
    ]:
        x, y = decoder_positions[
            key
        ]
        add_box(
            axis,
            x,
            y,
            box_w,
            box_h,
            key.upper(),
            shape_label(
                shapes[key]
            ),
        )
        add_arrow(
            axis,
            previous,
            (
                x,
                y + box_h / 2,
            ),
        )
        previous = (
            x + box_w,
            y + box_h / 2,
        )

    add_box(
        axis,
        15.55,
        6.3,
        box_w,
        box_h,
        "Output",
        "1 logit\n256×256",
    )
    add_arrow(
        axis,
        previous,
        (
            15.55,
            6.72,
        ),
    )

    skip_pairs = [
        ("e4", "d4"),
        ("e3", "d3"),
        ("e2", "d2"),
        ("e1", "d1"),
    ]

    for index, (
        encoder_key,
        decoder_key,
    ) in enumerate(skip_pairs):
        ex, ey = encoder_positions[
            encoder_key
        ]
        dx, dy = decoder_positions[
            decoder_key
        ]

        if attention:
            gate_x = (
                ex + dx
            ) / 2
            gate_y = max(
                ey,
                dy,
            ) + 1.0
            add_box(
                axis,
                gate_x - 0.43,
                gate_y - 0.28,
                0.86,
                0.56,
                f"AG{4-index}",
                "gated skip",
            )
            add_arrow(
                axis,
                (
                    ex + box_w / 2,
                    ey + box_h,
                ),
                (
                    gate_x - 0.43,
                    gate_y,
                ),
                connectionstyle="arc3,rad=-0.15",
                linestyle="--",
            )
            add_arrow(
                axis,
                (
                    gate_x + 0.43,
                    gate_y,
                ),
                (
                    dx + box_w / 2,
                    dy + box_h,
                ),
                connectionstyle="arc3,rad=-0.15",
                linestyle="--",
            )
        else:
            add_arrow(
                axis,
                (
                    ex + box_w / 2,
                    ey + box_h,
                ),
                (
                    dx + box_w / 2,
                    dy + box_h,
                ),
                connectionstyle="arc3,rad=-0.25",
                linestyle="--",
            )

    parameter_count = model_parameter_count(
        model
    )
    title = (
        "Attention U-Net"
        if attention
        else "U-Net"
    )
    modality_text = ", ".join(
        modality.title()
        for modality in modalities
    )

    axis.set_title(
        f"{title} — checkpoint-verified architecture\n"
        f"Applicable track(s): {modality_text} | "
        f"input channels: {in_ch} | "
        f"trainable parameters: {parameter_count:,}",
        fontsize=13,
        fontweight="bold",
    )

    base_name = (
        "architecture_attention_unet_exact"
        if attention
        else "architecture_unet_exact"
    )

    if config_label != "shared":
        base_name += (
            f"_{config_label}"
        )

    base_path = (
        ARCHITECTURE_DIR
        / base_name
    )
    save_figure_formats(
        figure,
        base_path,
    )
    plt.show()
    plt.close(figure)

    return {
        "model_name": model_name,
        "config_label": config_label,
        "modalities": list(
            modalities
        ),
        "model_config": config,
        "parameter_count": parameter_count,
        "figure_base": str(
            base_path
        ),
        "shapes": {
            key: list(value)
            for key, value
            in shapes.items()
        },
    }

architecture_records = []

if GENERATE_ARCHITECTURE_FIGURES:
    for model_name in [
        "unet",
        "attention_unet",
    ]:
        for (
            config_label,
            modalities,
            config,
        ) in configuration_groups(
            model_name
        ):
            architecture_records.append(
                draw_unet_architecture(
                    model_name,
                    config_label,
                    modalities,
                    config,
                )
            )

In [ ]:

def draw_swin_architecture(
    config_label: str,
    modalities: Sequence[str],
    config: Dict,
):
    model_name = "swin_tiny_unet"

    if config.get(
        "serialized_module",
        False,
    ):
        reference_modality = modalities[0]
        model = load_selected_model(
            CHECKPOINT_MAP[
                (
                    reference_modality,
                    model_name,
                    FIGURE_SEED,
                )
            ],
            model_name,
            CHECKPOINT_LOAD_INFO[
                (
                    reference_modality,
                    model_name,
                    FIGURE_SEED,
                )
            ],
        )
    else:
        model = build_model(
            model_name,
            config,
        )

    exact_mammography_family = (config.get("family") == MAMMOGRAPHY_ARCHITECTURE_FAMILY)
    final_decoder_modules = ["c0a", "c0b"] if exact_mammography_family else ["dec0"]

    available_modules = dict(model.named_modules())

    canonical_stage_names = [
        "features.1",
        "features.3",
        "features.5",
        "features.7",
    ]

    stage_name_map = {}

    for canonical_name in canonical_stage_names:
        candidate_names = [
            canonical_name,
            f"swin.{canonical_name}",
            f"backbone.{canonical_name}",
            f"encoder.{canonical_name}",
            f"model.{canonical_name}",
        ]

        resolved_name = next(
            (
                candidate
                for candidate in candidate_names
                if candidate in available_modules
            ),
            None,
        )

        if resolved_name is not None:
            stage_name_map[canonical_name] = resolved_name

    selected = [
        *stage_name_map.values(),
        "center",
        "dec3",
        "dec2",
        "dec1",
        *final_decoder_modules,
        "out",
    ]

    in_ch = int(
        config.get(
            "in_ch",
            model_input_channels(model),
        )
    )

    shapes = trace_selected_modules(
        model,
        selected,
        input_channels=in_ch,
    )

    normalized_shapes = {}
    reverse_stage_name_map = {
        actual_name: canonical_name
        for canonical_name, actual_name in stage_name_map.items()
    }

    for key, value in shapes.items():
        canonical_key = reverse_stage_name_map.get(key, key)

        if canonical_key.startswith("features.") and len(value) == 4:
            normalized_shapes[canonical_key] = (
                value[0],
                value[3],
                value[1],
                value[2],
            )
        else:
            normalized_shapes[canonical_key] = value

    if exact_mammography_family and "c0b" in normalized_shapes:
        normalized_shapes["dec0"] = normalized_shapes["c0b"]

    # Obtain decoder blocks from the actual forward pass.
    swin_stage_fallback = {
        "features.1": (1, 96, 64, 64),
        "features.3": (1, 192, 32, 32),
        "features.5": (1, 384, 16, 16),
        "features.7": (1, 768, 8, 8),
    }

    for stage_name, stage_shape in swin_stage_fallback.items():
        normalized_shapes.setdefault(stage_name, stage_shape)

    required = {
        "features.1",
        "features.3",
        "features.5",
        "features.7",
        "center",
        "dec3",
        "dec2",
        "dec1",
        "dec0",
    }

    missing_required = sorted(
        required.difference(normalized_shapes.keys())
    )

    if missing_required:
        raise RuntimeError(
            "The checkpoint-loaded Swin-Tiny U-Net is missing required "
            f"decoder blocks: {missing_required}. "
            f"Available: {sorted(normalized_shapes)}"
        )

    figure, axis = plt.subplots(
        figsize=(17.5, 7.2)
    )
    axis.set_xlim(
        0,
        18.0,
    )
    axis.set_ylim(
        0,
        8,
    )
    axis.axis("off")

    box_w = 1.35
    box_h = 0.9

    positions = {
        "input": (0.0, 6.2),
        "features.1": (1.8, 6.2),
        "features.3": (3.8, 4.9),
        "features.5": (5.8, 3.6),
        "features.7": (7.8, 2.3),
        "center": (9.8, 1.2),
        "dec3": (11.2, 2.3),
        "dec2": (12.7, 3.6),
        "dec1": (14.2, 4.9),
        "dec0": (15.7, 6.2),
        "output": (17.25, 6.2),
    }

    add_box(
        axis,
        *positions["input"],
        box_w,
        box_h,
        "Input",
        f"{in_ch} ch\n256×256",
    )

    labels = {
        "features.1": "Swin stage 1\ndepth 2",
        "features.3": "Swin stage 2\ndepth 2",
        "features.5": "Swin stage 3\ndepth 6",
        "features.7": "Swin stage 4\ndepth 2",
        "center": "Conv center",
        "dec3": "Decoder 3",
        "dec2": "Decoder 2",
        "dec1": "Decoder 1",
        "dec0": "Decoder 0",
    }

    path_order = [
        "input",
        "features.1",
        "features.3",
        "features.5",
        "features.7",
        "center",
        "dec3",
        "dec2",
        "dec1",
        "dec0",
        "output",
    ]

    for key in path_order[
        1:-1
    ]:
        add_box(
            axis,
            *positions[key],
            box_w,
            box_h,
            labels[key],
            shape_label(
                normalized_shapes[
                    key
                ]
            ),
        )

    add_box(
        axis,
        *positions["output"],
        box_w,
        box_h,
        "Output",
        "1 logit\n256×256",
    )

    for first, second in zip(
        path_order[:-1],
        path_order[1:],
    ):
        x1, y1 = positions[
            first
        ]
        x2, y2 = positions[
            second
        ]
        add_arrow(
            axis,
            (
                x1 + box_w,
                y1 + box_h / 2,
            ),
            (
                x2,
                y2 + box_h / 2,
            ),
        )

    for encoder_key, decoder_key in [
        ("features.5", "dec3"),
        ("features.3", "dec2"),
        ("features.1", "dec1"),
    ]:
        ex, ey = positions[
            encoder_key
        ]
        dx, dy = positions[
            decoder_key
        ]
        add_arrow(
            axis,
            (
                ex + box_w / 2,
                ey + box_h,
            ),
            (
                dx + box_w / 2,
                dy + box_h,
            ),
            connectionstyle="arc3,rad=-0.25",
            linestyle="--",
        )

    parameter_count = model_parameter_count(
        model
    )
    modality_text = ", ".join(
        modality.title()
        for modality in modalities
    )

    axis.set_title(
        "Swin-Tiny U-Net — checkpoint-verified architecture\n"
        f"Applicable track(s): {modality_text} | "
        f"input channels: {in_ch} | stage depths 2-2-6-2 | "
        f"trainable parameters: {parameter_count:,}",
        fontsize=13,
        fontweight="bold",
    )

    base_name = (
        "architecture_swin_tiny_unet_exact"
    )

    if config_label != "shared":
        base_name += (
            f"_{config_label}"
        )

    base_path = (
        ARCHITECTURE_DIR
        / base_name
    )
    save_figure_formats(
        figure,
        base_path,
    )
    plt.show()
    plt.close(figure)

    return {
        "model_name": model_name,
        "config_label": config_label,
        "modalities": list(
            modalities
        ),
        "model_config": config,
        "parameter_count": parameter_count,
        "figure_base": str(
            base_path
        ),
        "shapes": {
            key: list(value)
            for key, value
            in normalized_shapes.items()
        },
    }

if GENERATE_ARCHITECTURE_FIGURES:
    for (
        config_label,
        modalities,
        config,
    ) in configuration_groups(
        "swin_tiny_unet"
    ):
        architecture_records.append(
            draw_swin_architecture(
                config_label,
                modalities,
                config,
            )
        )

architecture_report = {
    "figure_seed": FIGURE_SEED,
    "article_expected_parameter_counts": ARTICLE_EXPECTED_PARAMS,
    "architectures": architecture_records,
    "checkpoint_verification": checkpoint_table.to_dict(
        orient="records"
    ),
}

(
    REPORT_DIR
    / "architecture_authenticity_report.json"
).write_text(
    json.dumps(
        architecture_report,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "Architecture authenticity report:",
    REPORT_DIR
    / "architecture_authenticity_report.json",
)

## 4. Manifest normalization and real NPZ loading


In [ ]:

def load_manifest(
    modality: str,
) -> pd.DataFrame:
    manifest_path = MANIFEST_PATHS[
        modality
    ]

    frame = pd.read_csv(
        manifest_path,
        dtype=str,
        keep_default_na=False,
    )

    for column in [
        "dataset",
        "split",
        "patient_id",
        "case_id",
        "sample_id",
        "npz_path",
    ]:
        if column not in frame.columns:
            frame[column] = ""

    frame["dataset"] = (
        frame["dataset"]
        .astype(str)
        .str.strip()
        .str.lower()
    )
    frame["split"] = (
        frame["split"]
        .astype(str)
        .str.strip()
        .str.lower()
    )
    frame["sample_id"] = (
        frame["sample_id"]
        .astype(str)
        .str.strip()
    )
    frame["patient_id"] = (
        frame["patient_id"]
        .astype(str)
        .str.strip()
    )

    if "global_patient_id" not in frame.columns:
        frame["global_patient_id"] = (
            frame["dataset"]
            + "::"
            + frame["patient_id"]
        )

    frame["modality_name"] = modality
    frame["manifest_path"] = str(
        manifest_path
    )

    return frame

MANIFESTS = {
    modality: load_manifest(
        modality
    )
    for modality in [
        "mammography",
        "mri",
        "ultrasound",
    ]
}

for modality, frame in MANIFESTS.items():
    print(
        "\n",
        modality,
        frame.groupby(
            [
                "dataset",
                "split",
            ]
        ).size(),
    )

def build_npz_index(
    search_root: Path,
):
    print(
        "Indexing NPZ files under:",
        search_root,
    )

    files = list(
        search_root.rglob(
            "*.npz"
        )
    )

    by_name = defaultdict(list)
    by_suffix = defaultdict(list)
    by_dataset_split_name = defaultdict(
        list
    )

    for path in files:
        by_name[
            path.name
        ].append(path)

        normalized = str(
            path
        ).replace(
            "\\",
            "/",
        )

        if "/npz/" in normalized:
            suffix = (
                "npz/"
                + normalized.split(
                    "/npz/",
                    1,
                )[1]
            )
            by_suffix[
                suffix
            ].append(path)

            suffix_parts = Path(
                suffix
            ).parts

            if len(
                suffix_parts
            ) >= 4:
                dataset_name = str(
                    suffix_parts[
                        -3
                    ]
                ).lower()
                split_name = str(
                    suffix_parts[
                        -2
                    ]
                ).lower()
                file_name = str(
                    suffix_parts[
                        -1
                    ]
                )

                by_dataset_split_name[
                    (
                        dataset_name,
                        split_name,
                        file_name,
                    )
                ].append(path)

        parts = path.parts

        if len(parts) >= 2:
            suffix = (
                f"{parts[-2]}/"
                f"{parts[-1]}"
            )
            by_suffix[
                suffix
            ].append(path)

    print(
        "Indexed NPZ files:",
        len(files),
    )

    return {
        "files": files,
        "by_name": by_name,
        "by_suffix": by_suffix,
        "by_dataset_split_name": (
            by_dataset_split_name
        ),
    }

NPZ_INDEX = {
    modality: build_npz_index(
        DATA_SEARCH_ROOTS[
            modality
        ]
    )
    for modality in DATA_SEARCH_ROOTS
}

def unique_existing(
    candidates: Iterable[Path],
    context: str,
) -> Optional[Path]:
    output = []
    seen = set()

    for candidate in candidates:
        candidate = Path(
            candidate
        )

        if not candidate.exists():
            continue

        try:
            key = str(
                candidate.resolve()
            )
        except Exception:
            key = str(
                candidate
            )

        if key not in seen:
            seen.add(key)
            output.append(
                candidate
            )

    if len(output) == 1:
        return output[0]

    if len(output) > 1:
        hashes = defaultdict(
            list
        )

        for path in output:
            hashes[
                sha256_file(
                    path
                )
            ].append(path)

        if len(hashes) == 1:
            return sorted(
                output,
                key=lambda path: str(
                    path
                ),
            )[0]

        raise RuntimeError(
            f"Ambiguous non-identical files for {context}: "
            f"{output[:20]}"
        )

    return None

def resolve_npz(
    modality: str,
    row: pd.Series,
) -> Path:
    manifest_root = DATA_ROOTS[
        modality
    ]
    search_root = DATA_SEARCH_ROOTS[
        modality
    ]
    index = NPZ_INDEX[
        modality
    ]

    stored_text = str(
        row["npz_path"]
    ).replace(
        "\\",
        "/",
    )
    stored_path = Path(
        stored_text
    )

    sample_name = (
        str(
            row["sample_id"]
        )
        if str(
            row["sample_id"]
        ).lower().endswith(
            ".npz"
        )
        else (
            f"{row['sample_id']}.npz"
        )
    )

    direct_candidates = [
        stored_path,
        manifest_root
        / stored_text,
        search_root
        / stored_text,
        manifest_root
        / sample_name,
        manifest_root
        / str(
            row["split"]
        )
        / sample_name,
    ]

    candidate = unique_existing(
        direct_candidates,
        (
            f"{modality}:"
            f"{row['sample_id']}:"
            "direct"
        ),
    )

    if candidate is not None:
        return candidate

    suffix_candidates = []

    if "/npz/" in stored_text:
        suffix = (
            "npz/"
            + stored_text.split(
                "/npz/",
                1,
            )[1]
        )
        suffix_candidates.extend(
            index[
                "by_suffix"
            ].get(
                suffix,
                [],
            )
        )

    split_suffix = (
        f"{row['split']}/"
        f"{stored_path.name}"
    )
    suffix_candidates.extend(
        index[
            "by_suffix"
        ].get(
            split_suffix,
            [],
        )
    )

    dataset_name = str(
        row["dataset"]
    ).lower()
    split_name = str(
        row["split"]
    ).lower()

    for file_name in {
        sample_name,
        stored_path.name,
    }:
        suffix_candidates.extend(
            index[
                "by_dataset_split_name"
            ].get(
                (
                    dataset_name,
                    split_name,
                    file_name,
                ),
                [],
            )
        )

    candidate = unique_existing(
        suffix_candidates,
        (
            f"{modality}:"
            f"{row['sample_id']}:"
            "suffix"
        ),
    )

    if candidate is not None:
        return candidate

    filename_candidates = (
        index[
            "by_name"
        ].get(
            sample_name,
            [],
        )
        + index[
            "by_name"
        ].get(
            stored_path.name,
            [],
        )
    )

    candidate = unique_existing(
        filename_candidates,
        (
            f"{modality}:"
            f"{row['sample_id']}:"
            "filename"
        ),
    )

    if candidate is not None:
        return candidate

    raise FileNotFoundError(
        f"NPZ not found for {modality}, "
        f"sample_id={row['sample_id']}, "
        f"dataset={row['dataset']}, "
        f"split={row['split']}, "
        f"stored_path={stored_text}, "
        f"manifest_root={manifest_root}, "
        f"search_root={search_root}"
    )

def normalize_image_array(
    image: np.ndarray,
) -> np.ndarray:
    image = image.astype(
        np.float32
    )

    if image.ndim == 2:
        image = image[
            None,
            ...,
        ]

    elif image.ndim == 3:
        if (
            image.shape[0]
            in [
                1,
                3,
            ]
        ):
            pass

        elif (
            image.shape[-1]
            in [
                1,
                3,
            ]
        ):
            image = image.transpose(
                2,
                0,
                1,
            )

        else:
            raise RuntimeError(
                f"Unsupported image shape: {image.shape}"
            )

    else:
        raise RuntimeError(
            f"Unsupported image dimensions: {image.shape}"
        )

    if image.shape[0] == 1:
        image = np.repeat(
            image,
            3,
            axis=0,
        )

    if image.shape[0] != 3:
        raise RuntimeError(
            "Expected 3 input channels after normalization, "
            f"got {image.shape}"
        )

    return image

def load_npz_sample(
    modality: str,
    row: pd.Series,
):
    path = resolve_npz(
        modality,
        row,
    )

    with np.load(
        path
    ) as data:
        if "image" not in data.files:
            raise RuntimeError(
                f"'image' key missing from {path}"
            )

        if "mask" not in data.files:
            raise RuntimeError(
                f"'mask' key missing from {path}"
            )

        image_raw = data[
            "image"
        ]
        mask = data[
            "mask"
        ].astype(
            np.uint8
        )

    image = normalize_image_array(
        image_raw
    )

    if mask.ndim == 3:
        mask = np.squeeze(
            mask
        )

    if mask.shape != (
        IMAGE_SIZE,
        IMAGE_SIZE,
    ):
        raise RuntimeError(
            f"Unexpected mask shape {mask.shape} in {path}"
        )

    mask = (
        mask > 0
    ).astype(
        np.uint8
    )

    if image.shape != (
        3,
        IMAGE_SIZE,
        IMAGE_SIZE,
    ):
        raise RuntimeError(
            f"Unexpected image shape {image.shape} in {path}"
        )

    return {
        "path": path,
        "image": image,
        "mask": mask,
        "image_raw_shape": list(
            image_raw.shape
        ),
    }

## 5. Archived metric discovery and verification


In [ ]:

def candidate_csv_files(
    modality: str,
) -> List[Path]:
    roots = []

    if RESULTS_ROOTS[
        modality
    ] is not None:
        roots.append(
            RESULTS_ROOTS[
                modality
            ]
        )

    roots.extend(
        search_roots()
    )

    output = []
    seen = set()

    for root in roots:
        if not root.exists():
            continue
        try:
            paths = list(
                root.rglob("*.csv")
            )
        except Exception:
            paths = []

        for path in paths:
            try:
                key = str(
                    path.resolve()
                )
            except Exception:
                key = str(path)

            if key in seen:
                continue

            normalized = str(
                path
            ).lower()

            if modality == "mammography":
                relevant = (
                    "detailed_metrics" in normalized
                )
            elif modality == "mri":
                relevant = (
                    "crop_level_metrics" in normalized
                )
            else:
                relevant = (
                    "validation_crop_metrics" in normalized
                    or "test_crop_metrics" in normalized
                    or "external_crop_metrics" in normalized
                )

            if relevant:
                seen.add(key)
                output.append(path)

    return output

def normalize_metrics_frame(
    frame: pd.DataFrame,
    modality: str,
    source_path: Path,
) -> pd.DataFrame:
    output = frame.copy()

    rename_map = {
        "model": "model_name",
        "empty_pred": "empty_prediction",
        "hd95": "hd95_px",
        "asd": "asd_px",
    }
    output = output.rename(
        columns=rename_map
    )

    if "model_name" not in output.columns:
        stem = source_path.name.lower()
        for model_name in MODEL_NAMES:
            if model_name in stem:
                output["model_name"] = model_name
                break

    if "seed" not in output.columns:
        match = re.search(
            r"seed[_-]?(42|123|2025)",
            source_path.name.lower(),
        )
        if match:
            output["seed"] = int(
                match.group(1)
            )

    if "split" not in output.columns:
        if "split_eval" in output.columns:
            output["split"] = output[
                "split_eval"
            ]
        else:
            normalized = str(
                source_path
            ).lower()
            if "validation" in normalized:
                output["split"] = "validation"
            elif "external" in normalized:
                output["split"] = "external"
            elif "test" in normalized:
                output["split"] = "test"

    if "threshold" not in output.columns:
        output["threshold"] = np.nan

    for column in [
        "sample_id",
        "patient_id",
        "model_name",
        "split",
    ]:
        if column not in output.columns:
            output[column] = ""
        output[column] = output[
            column
        ].astype(str)

    output["modality_name"] = modality
    output["archived_csv_path"] = str(
        source_path
    )

    return output

ARCHIVED_METRICS = {}

for modality in [
    "mammography",
    "mri",
    "ultrasound",
]:
    frames = []

    for path in candidate_csv_files(
        modality
    ):
        try:
            frame = pd.read_csv(path)
        except Exception:
            continue

        if (
            "sample_id"
            not in frame.columns
            or "dice"
            not in frame.columns
        ):
            continue

        frames.append(
            normalize_metrics_frame(
                frame,
                modality,
                path,
            )
        )

    if frames:
        merged = pd.concat(
            frames,
            ignore_index=True,
        )

        merged = merged.drop_duplicates(
            subset=[
                "sample_id",
                "model_name",
                "seed",
                "split",
                "threshold",
            ],
            keep="first",
        )

        ARCHIVED_METRICS[
            modality
        ] = merged

        print(
            modality,
            "archived case-level metrics:",
            len(merged),
        )
    else:
        ARCHIVED_METRICS[
            modality
        ] = pd.DataFrame()

        print(
            modality,
            "archived case-level metrics: not found",
        )

In [ ]:

def binary_counts(
    prediction: np.ndarray,
    target: np.ndarray,
):
    prediction = prediction.astype(
        bool
    )
    target = target.astype(
        bool
    )

    tp = int(
        np.logical_and(
            prediction,
            target,
        ).sum()
    )
    fp = int(
        np.logical_and(
            prediction,
            ~target,
        ).sum()
    )
    fn = int(
        np.logical_and(
            ~prediction,
            target,
        ).sum()
    )
    tn = int(
        np.logical_and(
            ~prediction,
            ~target,
        ).sum()
    )

    return tp, fp, fn, tn

def metrics_from_counts(
    tp,
    fp,
    fn,
    tn,
    smooth=1e-8,
):
    return {
        "dice": (
            2 * tp + smooth
        ) / (
            2 * tp
            + fp
            + fn
            + smooth
        ),
        "iou": (
            tp + smooth
        ) / (
            tp
            + fp
            + fn
            + smooth
        ),
        "precision": (
            tp + smooth
        ) / (
            tp
            + fp
            + smooth
        ),
        "recall": (
            tp + smooth
        ) / (
            tp
            + fn
            + smooth
        ),
    }

def error_map(
    prediction: np.ndarray,
    target: np.ndarray,
) -> np.ndarray:
    output = np.zeros_like(
        target,
        dtype=np.uint8,
    )
    output[
        (prediction == 1)
        & (target == 1)
    ] = 1
    output[
        (prediction == 1)
        & (target == 0)
    ] = 2
    output[
        (prediction == 0)
        & (target == 1)
    ] = 3
    return output


def load_frozen_model(
    modality: str,
    model_name: str,
    seed: int,
) -> Tuple[nn.Module, Path]:
    key = (
        modality,
        model_name,
        seed,
    )
    checkpoint_path = CHECKPOINT_MAP[
        key
    ]
    model = load_selected_model(
        checkpoint_path,
        model_name,
        CHECKPOINT_LOAD_INFO[
            key
        ],
    )

    actual_count = model_parameter_count(
        model
    )

    if (
        actual_count
        != ACTUAL_PARAMETER_COUNTS[
            key
        ]
    ):
        raise RuntimeError(
            "Parameter count changed between checkpoint discovery and "
            f"inference: {actual_count} != "
            f"{ACTUAL_PARAMETER_COUNTS[key]}"
        )

    model = model.to(
        DEVICE
    )
    model.eval()

    return model, checkpoint_path

def adapt_image_channels(
    image: np.ndarray,
    expected_channels: int,
) -> np.ndarray:
    if image.ndim != 3:
        raise RuntimeError(
            f"Expected CHW image, got {image.shape}"
        )

    if image.shape[0] == expected_channels:
        return image

    if expected_channels == 1:
        return image[
            :1
        ]

    if (
        expected_channels == 3
        and image.shape[0] == 1
    ):
        return np.repeat(
            image,
            3,
            axis=0,
        )

    raise RuntimeError(
        "The prepared image cannot be adapted to the checkpoint input: "
        f"prepared={image.shape[0]} channels, "
        f"expected={expected_channels}."
    )

@torch.inference_mode()
def predict_sample(
    model: nn.Module,
    image: np.ndarray,
    threshold: float,
):
    expected_channels = model_input_channels(
        model
    )
    adapted_image = adapt_image_channels(
        image,
        expected_channels,
    )

    tensor = torch.from_numpy(
        adapted_image[
            None,
            ...
        ]
    ).float().to(
        DEVICE
    )

    context = (
        torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        )
        if (
            USE_AMP_FOR_FIGURE_INFERENCE
            and DEVICE.type == "cuda"
        )
        else nullcontext()
    )

    with context:
        logits = model(
            tensor
        )

    probability = (
        torch.sigmoid(
            logits
        )
        .detach()
        .float()
        .cpu()
        .numpy()[
            0,
            0,
        ]
    )

    prediction = (
        probability
        >= threshold
    ).astype(
        np.uint8
    )

    return (
        probability,
        prediction,
    )

def find_archived_metric(
    modality: str,
    sample_id: str,
    model_name: str,
    seed: int,
    split: str,
    threshold: float,
) -> Optional[pd.Series]:
    frame = ARCHIVED_METRICS[
        modality
    ]

    if frame.empty:
        return None

    subset = frame[
        (frame["sample_id"] == str(sample_id))
        & (
            frame["model_name"]
            == model_name
        )
    ].copy()

    if "seed" in subset.columns:
        numeric_seed = pd.to_numeric(
            subset["seed"],
            errors="coerce",
        )
        subset = subset[
            numeric_seed == int(seed)
        ]

    if "split" in subset.columns:
        split_values = (
            subset["split"]
            .astype(str)
            .str.lower()
        )

        acceptable = {
            split.lower(),
        }

        if split == "external_inbreast":
            acceptable.update({
                "external",
                "external_inbreast",
            })
        elif split == "external":
            acceptable.update({
                "external",
            })

        subset = subset[
            split_values.isin(
                acceptable
            )
        ]

    if "threshold" in subset.columns:
        numeric_threshold = pd.to_numeric(
            subset["threshold"],
            errors="coerce",
        )

        threshold_matches = subset[
            np.isclose(
                numeric_threshold,
                threshold,
                atol=1e-6,
                equal_nan=False,
            )
        ]

        if len(
            threshold_matches
        ):
            subset = threshold_matches

    if len(subset) == 0:
        return None

    if len(subset) > 1:
        subset = subset.sort_values(
            "archived_csv_path"
        )

    return subset.iloc[0]

## 6. Representative case selection


In [ ]:

def target_frame(
    modality: str,
    split_role: str,
) -> pd.DataFrame:
    split_value = SPLIT_REGISTRY[
        modality
    ][
        split_role
    ]

    frame = MANIFESTS[
        modality
    ]

    output = frame[
        frame["split"]
        == split_value
    ].copy()

    if output.empty:
        raise RuntimeError(
            f"No rows found for {modality} / {split_role} "
            f"(split={split_value})"
        )

    return output.reset_index(
        drop=True
    )

def archived_swin_scores(
    modality: str,
    split_role: str,
) -> Optional[pd.DataFrame]:
    frame = ARCHIVED_METRICS[
        modality
    ]

    if frame.empty:
        return None

    split_value = SPLIT_REGISTRY[
        modality
    ][
        split_role
    ]

    subset = frame[
        (frame["model_name"] == "swin_tiny_unet")
    ].copy()

    if "seed" in subset.columns:
        subset = subset[
            pd.to_numeric(
                subset["seed"],
                errors="coerce",
            )
            == FIGURE_SEED
        ]

    acceptable = {
        split_value,
    }

    if split_role == "external":
        acceptable.add(
            "external"
        )

    subset = subset[
        subset["split"]
        .astype(str)
        .str.lower()
        .isin(acceptable)
    ]

    threshold = FROZEN_THRESHOLDS[
        modality
    ][
        "swin_tiny_unet"
    ][
        FIGURE_SEED
    ]

    if "threshold" in subset.columns:
        numeric_threshold = pd.to_numeric(
            subset["threshold"],
            errors="coerce",
        )
        exact = subset[
            np.isclose(
                numeric_threshold,
                threshold,
                atol=1e-6,
                equal_nan=False,
            )
        ]
        if len(exact):
            subset = exact

    subset["dice"] = pd.to_numeric(
        subset["dice"],
        errors="coerce",
    )

    subset = subset.dropna(
        subset=["dice"]
    )

    if subset.empty:
        return None

    return subset[
        [
            "sample_id",
            "patient_id",
            "dice",
        ]
    ].drop_duplicates(
        "sample_id"
    )

def scan_swin_scores(
    modality: str,
    split_role: str,
) -> pd.DataFrame:
    frame = target_frame(
        modality,
        split_role,
    )

    model, checkpoint_path = load_frozen_model(
        modality,
        "swin_tiny_unet",
        FIGURE_SEED,
    )

    threshold = FROZEN_THRESHOLDS[
        modality
    ][
        "swin_tiny_unet"
    ][
        FIGURE_SEED
    ]

    rows = []

    for _, row in tqdm(
        frame.iterrows(),
        total=len(frame),
        desc=(
            f"Scanning {modality} "
            f"{split_role} with frozen Swin"
        ),
    ):
        sample = load_npz_sample(
            modality,
            row,
        )
        _, prediction = predict_sample(
            model,
            sample["image"],
            threshold,
        )
        tp, fp, fn, tn = binary_counts(
            prediction,
            sample["mask"],
        )
        metrics = metrics_from_counts(
            tp,
            fp,
            fn,
            tn,
        )

        rows.append({
            "sample_id": str(
                row["sample_id"]
            ),
            "patient_id": str(
                row["patient_id"]
            ),
            "dice": metrics[
                "dice"
            ],
        })

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    scan_frame = pd.DataFrame(
        rows
    )
    scan_path = (
        REPORT_DIR
        / (
            f"{modality}_{split_role}_"
            "swin_seed42_scan.csv"
        )
    )
    scan_frame.to_csv(
        scan_path,
        index=False,
    )

    return scan_frame

def select_by_quantiles(
    scores: pd.DataFrame,
    modality: str,
) -> List[str]:
    scores = scores.sort_values(
        "dice"
    ).reset_index(
        drop=True
    )

    chosen = []
    used_patients = set()

    for quantile in SELECTION_QUANTILES:
        target_value = float(
            scores["dice"].quantile(
                quantile
            )
        )

        candidates = scores.assign(
            distance=(
                scores["dice"]
                - target_value
            ).abs()
        ).sort_values(
            [
                "distance",
                "sample_id",
            ]
        )

        selected_row = None

        for _, candidate in candidates.iterrows():
            sample_id = str(
                candidate["sample_id"]
            )
            patient_id = str(
                candidate.get(
                    "patient_id",
                    "",
                )
            )

            if sample_id in chosen:
                continue

            if (
                modality == "mri"
                and patient_id
                in used_patients
            ):
                continue

            selected_row = candidate
            break

        if selected_row is None:
            for _, candidate in candidates.iterrows():
                sample_id = str(
                    candidate["sample_id"]
                )
                if sample_id not in chosen:
                    selected_row = candidate
                    break

        if selected_row is None:
            raise RuntimeError(
                "Unable to select distinct representative cases."
            )

        chosen.append(
            str(
                selected_row[
                    "sample_id"
                ]
            )
        )
        used_patients.add(
            str(
                selected_row.get(
                    "patient_id",
                    "",
                )
            )
        )

    return chosen

SELECTED_SAMPLE_IDS = {}

for modality in [
    "mammography",
    "mri",
    "ultrasound",
]:
    SELECTED_SAMPLE_IDS[
        modality
    ] = {}

    for split_role in [
        "validation",
        "external",
    ]:
        fixed = FIXED_SAMPLE_IDS[
            modality
        ][
            split_role
        ]

        if (
            SELECTION_MODE == "fixed"
            and fixed
        ):
            selected = list(
                fixed
            )
        else:
            scores = archived_swin_scores(
                modality,
                split_role,
            )

            if scores is None:
                print(
                    f"Archived Swin scores unavailable for "
                    f"{modality}/{split_role}; scanning the frozen model."
                )
                scores = scan_swin_scores(
                    modality,
                    split_role,
                )

            selected = select_by_quantiles(
                scores,
                modality,
            )

        if len(selected) != len(
            SELECTION_QUANTILES
        ):
            raise RuntimeError(
                f"Expected {len(SELECTION_QUANTILES)} selected cases, "
                f"got {len(selected)}"
            )

        SELECTED_SAMPLE_IDS[
            modality
        ][
            split_role
        ] = selected

selection_path = (
    REPORT_DIR
    / "selected_article_cases.json"
)
selection_path.write_text(
    json.dumps(
        SELECTED_SAMPLE_IDS,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        SELECTED_SAMPLE_IDS,
        indent=2,
    )
)

## 7. Authentic internal and external segmentation figures


In [ ]:

PROVENANCE_ROWS = []
CAPTION_RECORDS = []

def display_channel(
    modality: str,
    image: np.ndarray,
) -> np.ndarray:
    if modality == "mri":
        return image[1]
    return image[0]

def row_for_sample(
    modality: str,
    split_role: str,
    sample_id: str,
) -> pd.Series:
    frame = target_frame(
        modality,
        split_role,
    )
    subset = frame[
        frame["sample_id"]
        == str(sample_id)
    ]

    if len(subset) != 1:
        raise RuntimeError(
            f"Expected one manifest row for {modality}/{sample_id}, "
            f"found {len(subset)}"
        )

    return subset.iloc[0]

def render_prediction_figure(
    modality: str,
    split_role: str,
    sample_ids: Sequence[str],
) -> Path:
    models = {}
    checkpoint_paths = {}

    for model_name in MODEL_NAMES:
        model, checkpoint_path = load_frozen_model(
            modality,
            model_name,
            FIGURE_SEED,
        )
        models[
            model_name
        ] = model
        checkpoint_paths[
            model_name
        ] = checkpoint_path

    is_mri = (
        modality == "mri"
    )

    column_names = (
        [
            "Pre-contrast",
            "Early post-contrast",
            "Late post-contrast",
            "Ground truth",
            "U-Net",
            "Attention U-Net",
            "Swin-Tiny U-Net",
            "Swin error map",
        ]
        if is_mri
        else [
            "Prepared ROI",
            "Ground truth",
            "U-Net",
            "Attention U-Net",
            "Swin-Tiny U-Net",
            "Swin error map",
        ]
    )

    figure, axes = plt.subplots(
        len(sample_ids),
        len(column_names),
        figsize=(
            2.4 * len(column_names),
            2.8 * len(sample_ids),
        ),
        squeeze=False,
    )

    case_caption_rows = []

    for row_index, sample_id in enumerate(
        sample_ids
    ):
        manifest_row = row_for_sample(
            modality,
            split_role,
            sample_id,
        )
        sample = load_npz_sample(
            modality,
            manifest_row,
        )
        image = sample["image"]
        target = sample["mask"]

        predictions = {}
        probabilities = {}
        case_metrics = {}

        for model_name in MODEL_NAMES:
            threshold = FROZEN_THRESHOLDS[
                modality
            ][
                model_name
            ][
                FIGURE_SEED
            ]

            probability, prediction = predict_sample(
                models[model_name],
                image,
                threshold,
            )

            tp, fp, fn, tn = binary_counts(
                prediction,
                target,
            )
            metrics = metrics_from_counts(
                tp,
                fp,
                fn,
                tn,
            )

            archived_row = (
                find_archived_metric(
                    modality=modality,
                    sample_id=str(
                        manifest_row[
                            "sample_id"
                        ]
                    ),
                    model_name=model_name,
                    seed=FIGURE_SEED,
                    split=str(
                        manifest_row[
                            "split"
                        ]
                    ),
                    threshold=threshold,
                )
                if VERIFY_AGAINST_ARCHIVED_METRICS
                else None
            )

            archived_dice = np.nan
            archived_source = ""
            dice_difference = np.nan
            verification_status = (
                "not_requested"
                if not VERIFY_AGAINST_ARCHIVED_METRICS
                else "not_available"
            )

            if archived_row is not None:
                archived_dice = float(
                    archived_row[
                        "dice"
                    ]
                )
                archived_source = str(
                    archived_row.get(
                        "archived_csv_path",
                        "",
                    )
                )
                dice_difference = abs(
                    metrics["dice"]
                    - archived_dice
                )

                verification_status = (
                    "pass"
                    if dice_difference
                    <= ARCHIVE_DICE_TOLERANCE
                    else "fail"
                )

                if (
                    STRICT_ARCHIVE_VERIFICATION
                    and verification_status
                    == "fail"
                ):
                    raise RuntimeError(
                        "Archived metric verification failed for "
                        f"{modality}/{sample_id}/{model_name}: "
                        f"new={metrics['dice']:.8f}, "
                        f"archived={archived_dice:.8f}, "
                        f"difference={dice_difference:.8g}"
                    )

            predictions[
                model_name
            ] = prediction
            probabilities[
                model_name
            ] = probability
            case_metrics[
                model_name
            ] = metrics

            PROVENANCE_ROWS.append({
                "modality": modality,
                "split_role": split_role,
                "manifest_split": str(
                    manifest_row[
                        "split"
                    ]
                ),
                "dataset": str(
                    manifest_row[
                        "dataset"
                    ]
                ),
                "sample_id": str(
                    manifest_row[
                        "sample_id"
                    ]
                ),
                "patient_id": str(
                    manifest_row[
                        "patient_id"
                    ]
                ),
                "case_id": str(
                    manifest_row[
                        "case_id"
                    ]
                ),
                "npz_path": str(
                    sample["path"]
                ),
                "npz_sha256": sha256_file(
                    sample["path"]
                ),
                "image_sha256": sha256_array(
                    image
                ),
                "target_sha256": sha256_array(
                    target
                ),
                "model_name": model_name,
                "seed": FIGURE_SEED,
                "threshold": threshold,
                "checkpoint_path": str(
                    checkpoint_paths[
                        model_name
                    ]
                ),
                "checkpoint_sha256": sha256_file(
                    checkpoint_paths[
                        model_name
                    ]
                ),
                "probability_sha256": sha256_array(
                    probability.astype(
                        np.float32
                    )
                ),
                "prediction_sha256": sha256_array(
                    prediction
                ),
                "dice": metrics[
                    "dice"
                ],
                "iou": metrics[
                    "iou"
                ],
                "precision": metrics[
                    "precision"
                ],
                "recall": metrics[
                    "recall"
                ],
                "archived_dice": archived_dice,
                "archived_csv_path": archived_source,
                "absolute_dice_difference": dice_difference,
                "archive_verification_status": verification_status,
            })

        if is_mri:
            images_to_show = [
                image[0],
                image[1],
                image[2],
                target,
                predictions["unet"],
                predictions[
                    "attention_unet"
                ],
                predictions[
                    "swin_tiny_unet"
                ],
                error_map(
                    predictions[
                        "swin_tiny_unet"
                    ],
                    target,
                ),
            ]
        else:
            images_to_show = [
                image[0],
                target,
                predictions["unet"],
                predictions[
                    "attention_unet"
                ],
                predictions[
                    "swin_tiny_unet"
                ],
                error_map(
                    predictions[
                        "swin_tiny_unet"
                    ],
                    target,
                ),
            ]

        for column_index, (
            axis,
            array,
            column_name,
        ) in enumerate(
            zip(
                axes[row_index],
                images_to_show,
                column_names,
            )
        ):
            axis.imshow(
                array
            )
            axis.axis("off")

            if row_index == 0:
                axis.set_title(
                    column_name,
                    fontsize=10,
                    fontweight="bold",
                )

        prediction_start = (
            4
            if is_mri
            else 2
        )

        for offset, model_name in enumerate(
            MODEL_NAMES
        ):
            metric_axis = axes[
                row_index,
                prediction_start + offset,
            ]
            metric_axis.set_xlabel(
                "Dice "
                f"{case_metrics[model_name]['dice']:.3f}\n"
                "P "
                f"{case_metrics[model_name]['precision']:.3f} | "
                "R "
                f"{case_metrics[model_name]['recall']:.3f}",
                fontsize=8,
            )

        axes[
            row_index,
            0,
        ].set_ylabel(
            str(
                manifest_row[
                    "sample_id"
                ]
            ),
            fontsize=8,
        )

        case_caption_rows.append(
            {
                "sample_id": str(
                    manifest_row[
                        "sample_id"
                    ]
                ),
                "patient_id": str(
                    manifest_row[
                        "patient_id"
                    ]
                ),
                "unet_dice": case_metrics[
                    "unet"
                ][
                    "dice"
                ],
                "attention_unet_dice": case_metrics[
                    "attention_unet"
                ][
                    "dice"
                ],
                "swin_tiny_unet_dice": case_metrics[
                    "swin_tiny_unet"
                ][
                    "dice"
                ],
            }
        )

    split_label = (
        "internal validation"
        if split_role == "validation"
        else "primary external validation"
    )

    figure.suptitle(
        f"{modality.title()} — {split_label}\n"
        f"Frozen seed {FIGURE_SEED}; "
        "model-specific validation-selected thresholds",
        fontsize=14,
        fontweight="bold",
    )
    figure.tight_layout(
        rect=[
            0,
            0,
            1,
            0.95,
        ]
    )

    output_directory = (
        INTERNAL_DIR
        if split_role == "validation"
        else EXTERNAL_DIR
    )

    base_path = (
        output_directory
        / (
            f"{modality}_{split_role}_"
            "three_model_predictions"
        )
    )

    save_figure_formats(
        figure,
        base_path,
    )
    plt.show()
    plt.close(figure)

    CAPTION_RECORDS.append({
        "figure_base": str(
            base_path
        ),
        "modality": modality,
        "split_role": split_role,
        "seed": FIGURE_SEED,
        "sample_ids": list(
            sample_ids
        ),
        "cases": case_caption_rows,
    })

    for model in models.values():
        del model

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return base_path.with_suffix(
        ".png"
    )

GENERATED_PREDICTION_FIGURES = {
    "validation": {},
    "external": {},
}

for modality in [
    "mammography",
    "mri",
    "ultrasound",
]:
    if GENERATE_INTERNAL_VALIDATION_FIGURES:
        GENERATED_PREDICTION_FIGURES[
            "validation"
        ][
            modality
        ] = render_prediction_figure(
            modality,
            "validation",
            SELECTED_SAMPLE_IDS[
                modality
            ][
                "validation"
            ],
        )

    if GENERATE_EXTERNAL_FIGURES:
        GENERATED_PREDICTION_FIGURES[
            "external"
        ][
            modality
        ] = render_prediction_figure(
            modality,
            "external",
            SELECTED_SAMPLE_IDS[
                modality
            ][
                "external"
            ],
        )

In [ ]:

def make_contact_sheet(
    paths: Sequence[Path],
    output_path: Path,
    title: str,
):
    images = [
        Image.open(path).convert(
            "RGB"
        )
        for path in paths
    ]

    target_width = max(
        image.width
        for image in images
    )

    resized = []

    for image in images:
        if image.width != target_width:
            new_height = round(
                image.height
                * target_width
                / image.width
            )
            image = image.resize(
                (
                    target_width,
                    new_height,
                ),
                Image.Resampling.LANCZOS,
            )
        resized.append(image)

    title_height = 80
    gap = 20

    canvas_height = (
        title_height
        + sum(
            image.height
            for image in resized
        )
        + gap
        * (
            len(resized)
            - 1
        )
    )

    canvas = Image.new(
        "RGB",
        (
            target_width,
            canvas_height,
        ),
        "white",
    )

    draw = ImageDraw.Draw(
        canvas
    )
    draw.text(
        (
            20,
            20,
        ),
        title,
        fill="black",
    )

    y = title_height

    for image in resized:
        canvas.paste(
            image,
            (
                0,
                y,
            ),
        )
        y += (
            image.height
            + gap
        )

    canvas.save(
        output_path,
        dpi=(
            300,
            300,
        ),
    )

    return output_path

if GENERATE_COMBINED_CONTACT_SHEETS:
    if GENERATED_PREDICTION_FIGURES[
        "validation"
    ]:
        make_contact_sheet(
            [
                GENERATED_PREDICTION_FIGURES[
                    "validation"
                ][
                    modality
                ]
                for modality in [
                    "mammography",
                    "mri",
                    "ultrasound",
                ]
            ],
            OUTPUT_ROOT
            / "figure_internal_validation_all_modalities.png",
            "Internal validation — authentic frozen-model predictions",
        )

    if GENERATED_PREDICTION_FIGURES[
        "external"
    ]:
        make_contact_sheet(
            [
                GENERATED_PREDICTION_FIGURES[
                    "external"
                ][
                    modality
                ]
                for modality in [
                    "mammography",
                    "mri",
                    "ultrasound",
                ]
            ],
            OUTPUT_ROOT
            / "figure_external_validation_all_modalities.png",
            "Primary external validation — authentic frozen-model predictions",
        )

print(
    "Combined figures generated under:",
    OUTPUT_ROOT,
)

## 8. Optional training curves from archived histories


In [ ]:

def history_candidates(
    modality: str,
) -> List[Path]:
    roots = []

    if RESULTS_ROOTS[
        modality
    ] is not None:
        roots.append(
            RESULTS_ROOTS[
                modality
            ]
        )

    roots.extend(
        search_roots()
    )

    output = []
    seen = set()

    for root in roots:
        if not root.exists():
            continue

        try:
            paths = list(
                root.rglob("*.csv")
            )
        except Exception:
            paths = []

        for path in paths:
            normalized = path.name.lower()
            if not any(
                token in normalized
                for token in [
                    "history",
                    "training_log",
                ]
            ):
                continue

            score = score_modality_path(
                path,
                modality,
            )
            if score < 0:
                continue

            key = str(
                path.resolve()
            )
            if key not in seen:
                seen.add(key)
                output.append(path)

    return output

def infer_model_seed_from_path(
    path: Path,
):
    normalized = str(
        path
    ).lower()

    model_name = None
    for candidate in MODEL_NAMES:
        if candidate in normalized:
            model_name = candidate
            break

    seed = None
    match = re.search(
        r"seed[_-]?(42|123|2025)",
        normalized,
    )
    if match:
        seed = int(
            match.group(1)
        )

    return model_name, seed

def plot_history_file(
    modality: str,
    path: Path,
):
    try:
        frame = pd.read_csv(path)
    except Exception:
        return None

    if frame.empty:
        return None

    model_name, seed = infer_model_seed_from_path(
        path
    )

    if seed is not None and seed != FIGURE_SEED:
        return None

    epoch_column = None
    for candidate in [
        "epoch",
        "Epoch",
    ]:
        if candidate in frame.columns:
            epoch_column = candidate
            break

    if epoch_column is None:
        frame = frame.reset_index().rename(
            columns={
                "index": "epoch",
            }
        )
        epoch_column = "epoch"

    loss_pairs = [
        (
            "train_loss",
            "val_loss",
        ),
        (
            "loss",
            "val_loss",
        ),
        (
            "train_epoch_loss",
            "validation_loss",
        ),
    ]

    selected_pair = None

    for train_column, val_column in loss_pairs:
        if (
            train_column in frame.columns
            and val_column in frame.columns
        ):
            selected_pair = (
                train_column,
                val_column,
            )
            break

    if selected_pair is None:
        return None

    figure, axis = plt.subplots(
        figsize=(7, 4.5)
    )

    axis.plot(
        frame[epoch_column],
        frame[
            selected_pair[0]
        ],
        label="Training loss",
    )
    axis.plot(
        frame[epoch_column],
        frame[
            selected_pair[1]
        ],
        label="Validation loss",
    )

    axis.set_xlabel(
        "Epoch"
    )
    axis.set_ylabel(
        "Loss"
    )
    axis.set_title(
        f"{modality.title()} training history"
        + (
            f" — {MODEL_LABELS.get(model_name, model_name)}"
            if model_name
            else ""
        )
        + (
            f", seed {seed}"
            if seed is not None
            else ""
        )
    )
    axis.legend()
    axis.grid(
        True,
        alpha=0.3,
    )

    output_path = (
        CURVES_DIR
        / (
            f"{modality}_"
            f"{model_name or 'model'}_"
            f"seed{seed or 'unknown'}_"
            "training_curve"
        )
    )

    save_figure_formats(
        figure,
        output_path,
    )
    plt.show()
    plt.close(figure)

    return output_path.with_suffix(
        ".png"
    )

TRAINING_CURVE_OUTPUTS = []

if GENERATE_TRAINING_CURVES:
    for modality in [
        "mammography",
        "mri",
        "ultrasound",
    ]:
        candidates = history_candidates(
            modality
        )

        ordered = sorted(
            candidates,
            key=lambda path: (
                "swin_tiny_unet"
                not in str(path).lower(),
                "seed42"
                not in str(path).lower(),
                len(str(path)),
            ),
        )

        for path in ordered:
            output = plot_history_file(
                modality,
                path,
            )
            if output is not None:
                TRAINING_CURVE_OUTPUTS.append(
                    output
                )
                break

print(
    "Training curve outputs:",
    [
        str(path)
        for path in TRAINING_CURVE_OUTPUTS
    ],
)

## 9. Final authenticity reports and downloadable package


In [ ]:

provenance_frame = pd.DataFrame(
    PROVENANCE_ROWS
)

provenance_path = (
    REPORT_DIR
    / "figure_case_authenticity_manifest.csv"
)
provenance_frame.to_csv(
    provenance_path,
    index=False,
)

if VERIFY_AGAINST_ARCHIVED_METRICS:
    available = provenance_frame[
        provenance_frame[
            "archive_verification_status"
        ].isin([
            "pass",
            "fail",
        ])
    ]

    failed = provenance_frame[
        provenance_frame[
            "archive_verification_status"
        ] == "fail"
    ]

    print(
        "Archived metric checks available:",
        len(available),
    )
    print(
        "Archived metric checks failed:",
        len(failed),
    )

    if (
        STRICT_ARCHIVE_VERIFICATION
        and len(failed)
    ):
        raise RuntimeError(
            "At least one archived metric verification failed."
        )

caption_lines = [
    "# Article figure captions generated from the frozen pipeline",
    "",
]

for record in CAPTION_RECORDS:
    split_label = (
        "internal validation"
        if record[
            "split_role"
        ] == "validation"
        else "primary external validation"
    )

    caption_lines.append(
        f"## {record['modality'].title()} — {split_label}"
    )
    caption_lines.append(
        ""
    )
    caption_lines.append(
        "Predictions were regenerated from the exact seed-"
        f"{record['seed']} checkpoints using the frozen "
        "model-specific validation thresholds. "
        "Rows correspond to the following sample IDs:"
    )
    caption_lines.append(
        ""
    )

    for case in record[
        "cases"
    ]:
        caption_lines.append(
            "- "
            f"`{case['sample_id']}` "
            f"(patient `{case['patient_id']}`): "
            f"U-Net Dice {case['unet_dice']:.3f}, "
            f"Attention U-Net Dice "
            f"{case['attention_unet_dice']:.3f}, "
            f"Swin-Tiny U-Net Dice "
            f"{case['swin_tiny_unet_dice']:.3f}."
        )

    caption_lines.append(
        ""
    )
    caption_lines.append(
        "The error map uses integer labels: "
        "0 background, 1 true positive, 2 false positive, "
        "and 3 false negative."
    )
    caption_lines.append(
        ""
    )

caption_path = (
    REPORT_DIR
    / "ARTICLE_FIGURE_CAPTIONS.md"
)
caption_path.write_text(
    "\n".join(
        caption_lines
    ),
    encoding="utf-8",
)

run_config = {
    "mammography_architecture_source_notebook": "02_TRAIN_ROI256_CBIS_INBREAST_3MODELS_3SEEDS_Kaggle_v2_ENGLISH_QC.ipynb",
    "mammography_architecture_family": MAMMOGRAPHY_ARCHITECTURE_FAMILY,
    "runtime": RUNTIME_NAME,
    "device": str(DEVICE),
    "torch_version": torch.__version__,
    "figure_seed": FIGURE_SEED,
    "selection_mode": SELECTION_MODE,
    "selection_quantiles": SELECTION_QUANTILES,
    "selected_sample_ids": SELECTED_SAMPLE_IDS,
    "frozen_thresholds": FROZEN_THRESHOLDS,
    "data_roots": {
        key: str(value)
        for key, value
        in DATA_ROOTS.items()
    },
    "checkpoint_roots": {
        key: str(value)
        for key, value
        in CHECKPOINT_ROOTS.items()
    },
    "results_roots": {
        key: (
            str(value)
            if value is not None
            else None
        )
        for key, value
        in RESULTS_ROOTS.items()
    },
    "archive_dice_tolerance": ARCHIVE_DICE_TOLERANCE,
    "strict_archive_verification": STRICT_ARCHIVE_VERIFICATION,
}

run_config_path = (
    REPORT_DIR
    / "RUN_CONFIG.json"
)
run_config_path.write_text(
    json.dumps(
        run_config,
        indent=2,
    ),
    encoding="utf-8",
)

checksum_rows = []

for path in OUTPUT_ROOT.rglob("*"):
    if (
        path.is_file()
        and path.suffix.lower()
        in [
            ".png",
            ".pdf",
            ".svg",
            ".csv",
            ".json",
            ".md",
        ]
    ):
        checksum_rows.append({
            "relative_path": str(
                path.relative_to(
                    OUTPUT_ROOT
                )
            ),
            "sha256": sha256_file(
                path
            ),
            "size_bytes": path.stat().st_size,
        })

checksum_frame = pd.DataFrame(
    checksum_rows
).sort_values(
    "relative_path"
)

checksum_path = (
    REPORT_DIR
    / "OUTPUT_SHA256_MANIFEST.csv"
)
checksum_frame.to_csv(
    checksum_path,
    index=False,
)

final_zip = (
    Path(
        "/kaggle/working/"
        "AUTHENTIC_ARTICLE_FIGURES_ALL_MODALITIES.zip"
    )
    if IN_KAGGLE
    else OUTPUT_ROOT.parent
    / "AUTHENTIC_ARTICLE_FIGURES_ALL_MODALITIES.zip"
)

with zipfile.ZipFile(
    final_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    allowZip64=True,
) as archive:
    for path in OUTPUT_ROOT.rglob("*"):
        if path.is_file():
            archive.write(
                path,
                path.relative_to(
                    OUTPUT_ROOT.parent
                ),
            )

print(
    "Authenticity manifest:",
    provenance_path,
)
print(
    "Caption file:",
    caption_path,
)
print(
    "Checksum manifest:",
    checksum_path,
)
print(
    "Downloadable ZIP:",
    final_zip,
)

display(
    provenance_frame[
        [
            "modality",
            "split_role",
            "sample_id",
            "model_name",
            "threshold",
            "dice",
            "archived_dice",
            "absolute_dice_difference",
            "archive_verification_status",
        ]
    ]
)